# Notebook 04 — Fine-tune Whisper (core) — Mozilla *Lost in Transcription*

**Purpose.** Fine-tune the notebook-03 winner (`whisper-large-v3-turbo`, language
token `jw`) on **Jember data only**, and decide **Gate 2**: did fine-tuning
actually beat zero-shot? No extra corpora here — data mixing is notebook 06.

**Analogy.** Notebook 03 tested a new hire with no training at all. This is the
real course, with real course material. If after the course they are no better
than before it, do not buy more course material (notebook 06) — first find out
what is wrong with the teaching.

## What is locked before this notebook starts

| decision | value | source |
|---|---|---|
| model | `openai/whisper-large-v3-turbo` | NB 03 elbow test — every cheaper model is provably worse |
| language token | `jw` | NB 03 Q2 — `id` vs `jw` is a statistical tie, `jw` picked because the Javanese side needs the repair |
| training target | **`text_folded`** | official dev set uses no Jember diacritics (28 Aug) |
| decode | greedy + `no_repeat_ngram_size=4` | NB 03 lost ~6 WER points to repetition loops |
| primary validation | **372 official dev clips** | the only in-distribution sample that exists |
| split | frozen, never regenerated | NB 02 |

> **The 0.8960 from NB 03 is NOT the Gate 2 comparator.** It was measured with an
> unguarded greedy decoder. This notebook decodes the dev sets **once with the
> un-fine-tuned turbo under this notebook's own decode settings** (cell 9) and
> uses that as the comparator. 0.8960 is reported only as a historical note.

## Two validation layers, deliberately different jobs

1. **Jember dev split** — 1,097 native segments, mean 5.4 s, same corpus as the
   training data. Cheap, big, used for checkpoint selection and progress
   tracking. A same-corpus proxy: it flatters the model.
2. **Official dev** — 372 clips, mean 24.9 s, real code-switched voice notes,
   10.2 % of them longer than Whisper's 30 s window. This is the distribution the
   test set is drawn from. Every "better" claim has to survive here.

**Dev is always native segments**, in every run and every ablation, so numbers
stay comparable. Only the *training* side changes when `SEGMENT_MODE` changes.

## Reference spaces — two numbers, one primary

The model is trained to emit `text_folded`. So on the Jember dev it is scored
twice: against `text_folded` (**primary** — same orthographic space as both the
target and the official dev) and against `text_official` (carried only for
continuity with notebook 03). On the official dev the reference is the
organiser's own text, normalised by `scorer.official_normalize` — that one is
the number that predicts the leaderboard.

## 0 — Dependencies

`transformers`, `torch`, `soundfile`, `librosa`, `huggingface_hub` all ship with
the Kaggle GPU image. Two optional packages change only *how* the optimizer or
the adapter is built, never what is measured:

- **bitsandbytes** — 8-bit AdamW. Full fine-tuning of turbo (809 M params) with
  fp32 master weights needs ~12.9 GB of gradient + optimizer state on a 16 GB T4
  before a single activation is stored. 8-bit Adam drops that to ~8 GB. Falls
  back to `adafactor` (built into transformers) if the import fails.
- **peft** — only for `TRAIN_VARIANT = "lora"`, which is an ablation, not the
  baseline. Not installed unless that variant is selected.

In [ ]:
!pip list 2>/dev/null | grep -Ei '^(transformers|torch|soundfile|librosa|huggingface-hub|accelerate|bitsandbytes|peft|datasets|pandas|numpy) ' || echo 'listing skipped'

## 1 — Knobs, output convention, path discovery

Everything that decides what this run *is* lives in one block. Content-based
discovery again — `jember-segments-16k` belongs to `hamzapratama`, the source
corpora to `thoriqfarras`, and `indonesian_dev` carries an extra directory level
inside its archive, so nothing may be addressed by a hardcoded depth.

In [ ]:
# ============================================================
# CELL 1 - KNOBS, OUTPUT CONVENTION, PATH DISCOVERY
# ============================================================
import os, sys, re, csv, json, time, math, random, shutil, inspect, hashlib, collections
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Set BEFORE torch is imported. Kaggle's "T4 x2" gives two GPUs, but HF Trainer
# in a notebook process falls back to nn.DataParallel, which replicates the model
# every step and does the loss reduction on GPU 0 - with gradient checkpointing
# and a 809 M-param model that is a reliable way to OOM the card doing the extra
# work, in exchange for maybe 1.3x throughput. One GPU, deterministically.
USE_BOTH_GPUS = False
if not USE_BOTH_GPUS:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np, pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)

# ============================================================
# KNOBS - the whole configuration of this run
# ============================================================

# "sanity" = the smoke run the brief REQUIRES before any long run: overfit 32
#            segments, prove checkpoint-resume, measure steps/s, and walk the
#            full eval + artifact path on a tiny subset. ~25 min GPU.
#            Writes to findings/04-whisper-core-sanity/.
# "train"  = the real run.
RUN_MODE = "train"

# Pull the last checkpoint from the HF repo and continue from it.
#   RUN_MODE="sanity" + RESUME_FROM_HF=True  -> the CROSS-SESSION resume proof
#                                               (fresh session, after a sanity
#                                               run has pushed training state)
#   RUN_MODE="train"  + RESUME_FROM_HF=True  -> continue a training run that a
#                                               12 h session cut off
RESUME_FROM_HF = False

# Which stages of the sanity suite to run. Each is expensive enough that
# repeating a stage already proven is worth avoiding, and the results of earlier
# sessions are carried forward from the HF copy of sanity_report.json, so the
# final report is complete even when a session only ran one stage.
#   run 1 (fresh)          : ["overfit", "resume", "speed"]     RESUME_FROM_HF=False
#   run 2 (fresh session)  : ["cross_session"]                  RESUME_FROM_HF=True
#   run 3                  : RUN_MODE="train", this is ignored
# "all" is also accepted.
SANITY_STAGES = ["cross_session"]

# One variable per run (brief, section 4). "full" + "native" is the baseline;
# the rest are the ablation slots. A non-baseline run writes to its own
# findings/ folder and its own HF repo, so it can never overwrite the baseline
# it is meant to be compared against.
TRAIN_VARIANT = "full"        # "full" | "freeze_encoder" | "freeze_encoder_half" | "lora"
SEGMENT_MODE  = "concat25"    # "native" (5.4 s Jember segments) | "concat25" (glued to ~25 s)

# Free-form suffix for a run that changes the RECIPE rather than the variant or
# the segment mode - epochs, learning rate, batch. Without it such a run reuses
# the same RUN_ID and silently overwrites the findings folder and the HF repo of
# the run it is supposed to be compared against. Keep it short and factual.
RUN_NOTE = "1epoch"           # "" for the canonical run of this variant

# Locked by notebook 03. Do not edit without a new notebook 03 run.
HF_MODEL   = "openai/whisper-large-v3-turbo"
LANG_CODE  = "jw"             # Whisper's Javanese code is "jw", not ISO "jv"
LANG_NAME  = "javanese"       # what WhisperTokenizer wants as language=

# Training recipe. Starting point from the brief, to be adjusted after run one.
# 31 Aug: the official-dev curve went flat after ~25 of 273 steps while the
# Jember dev kept falling for another 16 points. Everything past that point
# traded Javanese accuracy for Indonesian accuracy at roughly par. This run
# tests whether one epoch reaches the same total WER without paying that tax -
# and if it does, every NB 06 mixing experiment becomes 3x cheaper.
NUM_EPOCHS        = 1
LEARNING_RATE     = 6e-6      # turbo / medium tier; 1e-5 would be for small
WARMUP_RATIO      = 0.10
PER_DEVICE_BATCH  = 4         # drop to 2 (and raise GRAD_ACCUM to 8) on OOM
GRAD_ACCUM        = 4         # effective batch 16
# "auto" targets ~8 eval points across the whole run, resolved in cell 4 once the
# item count is known. The brief's "every 200-400 steps" was written before the
# step count was: at effective batch 16 the native run is only ~1,040 steps, so
# 300 would buy 3 eval points - too few to select a checkpoint on, and too few
# for early stopping to mean anything.
EVAL_EVERY        = "auto"    # or an int, in optimizer steps
EVAL_SUBSET_N     = 300       # fixed Jember dev subset used during training

# Also evaluate the OFFICIAL dev during training, not only at the end.
# The 30 Aug run measured the official layer exactly once, at the final step,
# and so could not answer the question its own result raised: Jember dev fell
# 40 points while the official dev did not move, so did the official layer peak
# early and then regress? A curve answers that; an endpoint cannot.
# ~150 clips x ~1 s, at every eval point: about 20 min across a 3-epoch run.
OFFICIAL_EVAL_DURING_TRAINING = True
OFFICIAL_EVAL_N   = 150
# Checkpoint selection deliberately stays on the JEMBER dev metric. The official
# 372 are the only in-distribution validation that will ever exist, and choosing
# a checkpoint by them starts spending them as a tuning signal. The official
# curve here is an observation, not a selector.
SAVE_TOTAL_LIMIT  = 3
EARLY_STOP_PATIENCE = 3       # evals without improvement
OPTIM             = "auto"    # "auto" -> adamw_bnb_8bit if available, else adafactor
GRAD_CHECKPOINT   = True
SPEC_AUGMENT      = True

# Wall-clock guard. The brief: measure steps/s on the real configuration, then
# extrapolate BEFORE starting. Kaggle sessions die at ~12 h.
MAX_TRAIN_HOURS   = 10.0
ACCEPT_LONG_RUN   = False     # set True to override the projection abort

# Decode settings for EVERY evaluation in this notebook - fine-tuned model and
# zero-shot comparator alike. NB 03 proved an unguarded greedy decoder loses
# ~6 WER points to repetition loops, so the guard is on by default here.
NO_REPEAT_NGRAM   = 4
# NOT tied to SEGMENT_MODE. Concatenated items are training targets only - what
# gets generated is always a native Jember segment or an official clip, in every
# run. Making this depend on the segment mode would have given the concat25
# ablation a different decode setting from the baseline it is compared against,
# for no reason.
GEN_MAX_LENGTH    = 225
EVAL_BATCH        = 8

# How clips longer than Whisper's 30 s window are decoded. 10.2 % of the official
# dev is over 30 s, so this is not an edge case.
#   "chunk"     - fixed windows with a small overlap, duplicate words trimmed at
#                 the seam. Deterministic, version-independent, identical for
#                 every model.
#   "long_form" - transformers' own sequential long-form algorithm.
#   "auto"      - try long_form, fall back to chunk per clip.
# Default is "chunk" on purpose. The 29 Aug sanity run had the zero-shot pass
# decode 4 long clips with long_form while the fine-tuned pass fell back to
# chunks on the SAME clips (transformers 5.0.0 raised
# "TypeError: slice indices must be integers"). Two sides of a Gate 2 comparison
# running different decode algorithms is a contaminated comparison, and "auto"
# makes that failure silent and per-clip. A slightly cruder decode applied
# identically everywhere beats a better one applied inconsistently.
LONG_MODE         = "chunk"
LONG_CHUNK_S      = 28.0      # window length for "chunk"
LONG_OVERLAP_S    = 2.0       # seam overlap, trimmed by word-level dedup

# Decode all 208 recordings once per session into RAM as float16 (~1.2 GB).
# This is the brief's "cache the features once, do not recompute them every
# epoch", moved one step earlier in the pipeline. Caching log-Mel instead would
# be 4.3 GB on disk (128 mel bins x 3000 frames x 5,600 segments), while the mel
# transform itself is ~30 ms per 30 s clip against a ~1.5 s training step. The
# expensive part is the FLAC decode, and that is what this removes.
PRELOAD_AUDIO = True

# How often a RESUMABLE checkpoint (weights + optimizer state, several GB) is
# pushed to the HF training-state branch. Pushing on every save would spend more
# wall-clock uploading than training.
PUSH_EVERY_N_SAVES = 3

SEED = 20260829
random.seed(SEED); np.random.seed(SEED)

# ============================================================
FINDINGS = {}
# Re-running THIS cell discards everything the later cells collected. Run All.

def rec(section, **kw):
    FINDINGS.setdefault(section, {}).update(kw)

def head(title):
    print("\n" + "=" * 74); print(title); print("=" * 74)

def _num(x):
    # SIGNIFICANT digits, not decimal places. round(x, 4) reads fine for a WER
    # but turns a learning rate of 6e-06 into 0.0 - which is exactly what the
    # 29 Aug sanity digest reported, for every small number in the run.
    x = float(x)
    if x == 0.0 or not math.isfinite(x):
        return x
    return float("%.6g" % x)

def jsonable(o):
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return _num(o)
    if isinstance(o, (np.bool_,)):    return bool(o)
    if isinstance(o, float):          return _num(o)
    if isinstance(o, dict):  return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)): return [jsonable(v) for v in o]
    if isinstance(o, Path): return str(o)
    return o

# ============================================================
# OUTPUT CONVENTION - mirrored tree, same rule as notebooks 02 and 03.
# /kaggle/working/findings/<ID>/  <->  <repo>/findings/<ID>/
# A run that is not the baseline gets its own suffix, so it can never be
# mistaken for - or overwrite - the baseline it exists to be compared against.
# ============================================================
NOTEBOOK_ID = "04-whisper-core"
IS_BASELINE = (TRAIN_VARIANT == "full" and SEGMENT_MODE == "native")
if RUN_MODE == "sanity":
    RUN_TAG = "-sanity"
elif IS_BASELINE:
    RUN_TAG = ""
else:
    RUN_TAG = "-" + TRAIN_VARIANT + ("" if SEGMENT_MODE == "native" else "-" + SEGMENT_MODE)
# BASE_RUN_ID is the same variant and segment mode without the recipe note. The
# zero-shot comparator depends only on the decode settings, so a recipe variant
# can borrow the base run's cached comparator instead of spending ~25 min
# recomputing an identical number - the decode fingerprint check in cell 11 is
# what makes that safe.
BASE_RUN_ID = NOTEBOOK_ID + RUN_TAG
if RUN_MODE != "sanity" and RUN_NOTE:
    RUN_TAG = RUN_TAG + "-" + RUN_NOTE
RUN_ID = NOTEBOOK_ID + RUN_TAG

WORKING  = Path("/kaggle/working")
OUT_DIR  = WORKING / "findings" / RUN_ID

# Checkpoints go to /kaggle/temp, NOT under /kaggle/working. Everything inside
# /kaggle/working is the notebook's output: it is what "Download all" packages
# and what a committed run saves. A turbo checkpoint is ~3 GB of fp32 weights
# plus optimizer state, times save_total_limit, times the sanity suite's several
# trainers - which is how a run that produces 300 KB of findings ends up
# offering a 16 GB download. /kaggle/temp is scratch on the same disk and is
# never collected as output. Durability is the HF repo's job, not the output
# tab's, so nothing is lost by keeping weights out of it.
# 29 Aug: /kaggle/temp did NOT exist on the image, the fallback landed on
# /kaggle/working, and the run still offered a 9.7 GB download. Requiring the
# directory to pre-exist was the mistake - it just needs to be created, and it
# only has to satisfy one property: not be inside the output tree.
def _pick_scratch():
    for cand in (Path("/kaggle/temp"), Path("/tmp/nb-scratch"), Path.home() / "nb-scratch"):
        try:
            cand.mkdir(parents=True, exist_ok=True)
            probe = cand / ".write_probe"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink()
            if WORKING not in cand.parents and cand != WORKING:
                return cand
        except Exception:
            continue
    return WORKING

SCRATCH  = _pick_scratch()
CKPT_DIR = SCRATCH / "ckpt" / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

def out(filename):
    p = OUT_DIR / filename
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

_ORIG_STDOUT = getattr(sys, "_nb_orig_stdout", sys.stdout)
sys._nb_orig_stdout = _ORIG_STDOUT

class _Tee:
    def __init__(self, path, stream):
        self.f = open(path, "w", encoding="utf-8"); self.s = stream
    def write(self, t):
        self.s.write(t); self.f.write(t); return len(t)
    def flush(self):
        try: self.s.flush()
        except Exception: pass
        self.f.flush()
    def isatty(self): return False

LOG_PATH = out("run-log.txt")
try:
    if isinstance(sys.stdout, _Tee): sys.stdout.f.close()
except Exception:
    pass
sys.stdout = _Tee(LOG_PATH, _ORIG_STDOUT)

print("=" * 74)
print("RUN MODE  : %s        variant=%s   segments=%s" % (RUN_MODE.upper(), TRAIN_VARIANT, SEGMENT_MODE))
print("RUN ID    : %s   (baseline run: %s)" % (RUN_ID, IS_BASELINE))
print("RESUME    : %s" % RESUME_FROM_HF)
print("=" * 74)
if RUN_MODE == "sanity":
    print("!! SANITY / SMOKE RUN. Nothing below is a result. It proves the three")
    print("!! things the brief requires before a long run: the label pipeline can")
    print("!! overfit, checkpoint-resume really restores optimizer state, and the")
    print("!! projected wall-clock fits a session. Output -> findings/%s/." % RUN_ID)
print("output dir : %s   (this is what Download all packages)" % OUT_DIR)
print("ckpt dir   : %s" % CKPT_DIR)
if SCRATCH == WORKING:
    print("!! No writable scratch outside the output tree, so checkpoints land")
    print("!! inside it. Several GB will be offered as a download. Run")
    print("!!   !rm -rf %s" % CKPT_DIR)
    print("!! before saving the session, but only AFTER cell 15 has pushed to HF.")
else:
    print("             (scratch, outside the output tree - nothing here is downloaded)")
print("console log: %s" % LOG_PATH)
print("seed       : %d" % SEED)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device     : %s  (n_gpu=%d)" % (DEVICE, torch.cuda.device_count() if DEVICE == "cuda" else 0))
if DEVICE == "cuda":
    _p = torch.cuda.get_device_properties(0)
    print("gpu        : %s, %.1f GB" % (_p.name, _p.total_memory / 1e9))
else:
    print("!!! No CUDA device. Select a GPU accelerator before running anything")
    print("!!! past a syntax check - turbo on CPU is not a thing that finishes.")
import transformers
print("torch %s   transformers %s" % (torch.__version__, transformers.__version__))

INPUT = Path("/kaggle/input")
head("DISCOVERY")

# --- jember-segments-16k: marker is recordings/_meta/manifest.csv next to scorer.py
manifest_hits = sorted(INPUT.glob("**/recordings/_meta/manifest.csv"))
if not manifest_hits:
    manifest_hits = [p for p in INPUT.glob("**/manifest.csv")
                     if (p.parent / "scorer.py").exists() and (p.parent / "lexicon_v1.csv").exists()]
META_ROOT  = manifest_hits[0].parent if manifest_hits else None
AUDIO_ROOT = META_ROOT.parent.parent if META_ROOT else None

# --- indonesian_dev: the archive root carries an extra indonesian_dev/ level
dev_hits = sorted(INPUT.glob("**/indonesian_dev/metadata.tsv"))
if not dev_hits:
    dev_hits = [p for p in INPUT.glob("**/metadata.tsv") if (p.parent / "clips").is_dir()]
ODEV_META  = dev_hits[0] if dev_hits else None
ODEV_CLIPS = (ODEV_META.parent / "clips") if ODEV_META else None

# ==================================================================
# MANUAL OVERRIDE - only if a line below says NOT FOUND.
# ==================================================================
# META_ROOT  = Path("/kaggle/input/datasets/hamzapratama/jember-segments-16k/recordings/_meta")
# AUDIO_ROOT = META_ROOT.parent.parent
# ODEV_META  = Path("/kaggle/input/datasets/hamzapratama/indonesian-dev/indonesian_dev/metadata.tsv")
# ODEV_CLIPS = ODEV_META.parent / "clips"

print("META_ROOT  =", META_ROOT or "NOT FOUND")
print("AUDIO_ROOT =", AUDIO_ROOT or "NOT FOUND")
print("ODEV_META  =", ODEV_META or "NOT FOUND  <- attach the indonesian_dev dataset")
print("ODEV_CLIPS =", ODEV_CLIPS or "NOT FOUND")
if META_ROOT is None:
    print("")
    print("what IS mounted under /kaggle/input:")
    for _p in sorted(INPUT.glob("*")):
        print("   ", _p)
        for _q in sorted(_p.glob("*"))[:6]:
            print("      ", _q.name)
    # Raise HERE, where the problem is. Letting this cell finish only moves the
    # failure to "import scorer" two cells later, which reads like a missing
    # package rather than a missing dataset.
    raise RuntimeError(
        "jember-segments-16k is not mounted. Add data -> attach the dataset that "
        "holds recordings/_meta/manifest.csv (and indonesian_dev alongside it), "
        "then Run All. If it IS attached and the glob still misses, fill the "
        "MANUAL OVERRIDE block above and re-run this cell.")
else:
    print("")
    print("recordings/_meta/ :", sorted(p.name for p in META_ROOT.iterdir()))
    print("recordings/        : %d flac" % len(list((AUDIO_ROOT / "recordings").glob("*.flac"))))
if ODEV_CLIPS is not None:
    print("official dev clips : %d mp3" % len(list(ODEV_CLIPS.glob("*.mp3"))))

rec("env", notebook=NOTEBOOK_ID, run_id=RUN_ID, run_mode=RUN_MODE, seed=SEED,
    is_baseline=IS_BASELINE, train_variant=TRAIN_VARIANT, segment_mode=SEGMENT_MODE,
    resume_from_hf=RESUME_FROM_HF, python=sys.version.split()[0],
    pandas=pd.__version__, numpy=np.__version__, torch=torch.__version__,
    transformers=transformers.__version__, device=DEVICE,
    n_gpu=(torch.cuda.device_count() if DEVICE == "cuda" else 0),
    use_both_gpus=USE_BOTH_GPUS, meta_root=str(META_ROOT), audio_root=str(AUDIO_ROOT),
    odev_meta=str(ODEV_META), hf_model=HF_MODEL, lang=LANG_CODE)
rec("recipe", num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE, warmup_ratio=WARMUP_RATIO,
    per_device_batch=PER_DEVICE_BATCH, grad_accum=GRAD_ACCUM,
    effective_batch=PER_DEVICE_BATCH * GRAD_ACCUM, eval_every=EVAL_EVERY,
    eval_subset_n=EVAL_SUBSET_N,
    official_eval_during_training=OFFICIAL_EVAL_DURING_TRAINING,
    official_eval_n=OFFICIAL_EVAL_N, optim_request=OPTIM, grad_checkpoint=GRAD_CHECKPOINT,
    spec_augment=SPEC_AUGMENT, no_repeat_ngram=NO_REPEAT_NGRAM,
    gen_max_length=GEN_MAX_LENGTH, max_train_hours=MAX_TRAIN_HOURS,
    long_mode=LONG_MODE, long_chunk_s=LONG_CHUNK_S, long_overlap_s=LONG_OVERLAP_S)

## 2 — `scorer.py` from the mount, and the two reference spaces

Imported straight from `jember-segments-16k`, never rewritten: this notebook has
to score with exactly the module notebook 02 shipped and verified (28/28
self-tests, both normalizer bugs fixed).

`scorer.tokenize` runs `official_normalize` on both sides but does **not** fold
diacritics — folding is a separate view. So the folded space is scored by folding
both sides before handing them over.

In [ ]:
# ============================================================
# CELL 2 - SCORER + REFERENCE SPACES
# ============================================================
head("SCORER")
# scorer.py is not a package - it is a file inside the mounted dataset. A
# ModuleNotFoundError here almost always means the mount is wrong, not that
# something needs installing.
if META_ROOT is None or not (META_ROOT / "scorer.py").exists():
    raise RuntimeError(
        "scorer.py is not at META_ROOT (%s), so cell 1's discovery did not find "
        "jember-segments-16k. Right panel -> Input -> Add Input, attach BOTH "
        "'jember-segments-16k' and 'indonesian_dev', then Run All. Re-importing "
        "the .ipynb into Kaggle is the usual cause: an import makes a NEW "
        "notebook, and a new notebook has nothing attached." % META_ROOT)
sys.path.insert(0, str(META_ROOT))
import scorer
print("scorer version:", scorer.VERSION)
ok, checks = scorer.self_test(verbose=False)
print("self_test: %d/%d passed" % (sum(1 for _, c, _ in checks if c), len(checks)))
if not ok:
    for name, cond, detail in checks:
        if not cond:
            print("  FAILED:", name, detail)
    raise RuntimeError("scorer.py self-test failed - no WER below is trustworthy until this is green.")

def fold(s):
    # The folded space: same normalizer, diacritics collapsed. Applied to BOTH
    # sides. official_normalize is unaffected by the folding (it never keys off
    # an accented letter), so fold-then-normalize and normalize-then-fold agree.
    return scorer.fold_diacritics(s if isinstance(s, str) else "")

def wer_pack(refs, hyps, lex, prefix=""):
    # Every metric the brief asks for, for one (refs, hyps) pair, in one dict.
    m = scorer.wer_by_origin(refs, hyps, lex)
    jv_rate, n_jv_sub, n_jv_to_id, _pairs = jv_class_substitutions(refs, hyps, lex)
    n_ref_tok = sum(len(scorer.tokenize(r)) for r in refs)
    n_hyp_tok = sum(len(scorer.tokenize(h)) for h in hyps)
    n_digit = sum(1 for h in hyps if re.search(r"[0-9]", h or ""))
    d = dict(
        wer=round(m["wer_total"], 4), wer_jv=round(m["wer_jv"], 4),
        wer_id=round(m["wer_id"], 4), wer_both=round(m["wer_both"], 4),
        wer_unk=round(m["wer_unk"], 4),
        sub_rate=round(m["sub_rate"], 4), del_rate=round(m["del_rate"], 4),
        ins_rate=round(m["ins_rate"], 4), cosmetic_sub_pct=m["cosmetic_sub_pct"],
        jv_to_id_sub_pct=round(100 * jv_rate, 2), n_jv_sub=n_jv_sub, n_jv_to_id=n_jv_to_id,
        len_ratio=round(n_hyp_tok / max(n_ref_tok, 1), 4),
        pct_hyp_with_digit=round(100.0 * n_digit / max(len(hyps), 1), 2),
        n_utts=m["n_utts"], n_ref_tokens=m["n_ref_tokens"])
    return {(prefix + k): v for k, v in d.items()} if prefix else d

def jv_class_substitutions(refs, hyps, lex):
    # Identical to notebook 03's definition, so the 30.15 % guard number is
    # comparable: among substitutions whose REFERENCE token is lexicon-class
    # "jv", how many landed on a class-"id" hypothesis token. Deletions of a
    # Javanese word are a different failure and stay out of this rate.
    pairs = collections.Counter()
    n_jv_sub = n_jv_to_id = 0
    for r, h in zip(refs, hyps):
        rt, ht = scorer.tokenize(r), scorer.tokenize(h)
        for op, ri, hi in scorer.align_words(rt, ht):
            if op != "sub" or ri is None:
                continue
            ref_tok = rt[ri]
            if lex.get(scorer.lex_key(ref_tok), "unk") != "jv":
                continue
            n_jv_sub += 1
            hyp_tok = ht[hi]
            pairs[(ref_tok, hyp_tok)] += 1
            if lex.get(scorer.lex_key(hyp_tok), "unk") == "id":
                n_jv_to_id += 1
    return (n_jv_to_id / n_jv_sub if n_jv_sub else 0.0), n_jv_sub, n_jv_to_id, pairs

rec("scorer", version=scorer.VERSION, self_test_passed=ok, n_checks=len(checks))

# Historical numbers from notebook 03, carried only so the writeup can quote
# them. NONE of these is the Gate 2 comparator - cell 9 measures that here.
NB03 = dict(wer_official_unguarded=0.8960, wer_folded_unguarded=0.8719,
            jv_to_id_sub_pct=30.15, model="whisper-large-v3-turbo", lang="id_or_jw_tie")
print("")
print("NB 03 historical (unguarded greedy decoder, NOT the comparator):", NB03)
rec("nb03_historical", **NB03)

## 3 — Manifest, split, lexicon, both dev sets

The Jember side comes from the frozen manifest. The official side is read from
`metadata.tsv` and normalised with the same `scorer.official_normalize`, so both
references live in the same space the model is scored in.

Sanity checks demanded by the brief: 372 clips, 372 metadata rows, filenames
matching one-to-one.

In [ ]:
# ============================================================
# CELL 3 - MANIFEST / SPLIT / LEXICON / BOTH DEV SETS
# ============================================================
head("LOAD - JEMBER")
MANIFEST = pd.read_csv(META_ROOT / "manifest.csv", dtype=str, keep_default_na=False)
SPLIT    = pd.read_csv(META_ROOT / "split.csv", dtype=str, keep_default_na=False)
LEX      = scorer.load_lexicon(META_ROOT / "lexicon_v1.csv")
for c in ("start_sample", "end_sample"):
    MANIFEST[c] = MANIFEST[c].astype(int)
MANIFEST["dur_s"] = MANIFEST["dur_s"].astype(float)

TRAIN = MANIFEST[MANIFEST["split"] == "train"].sort_values(["audio_filename", "start_sample"]).reset_index(drop=True)
DEV   = MANIFEST[MANIFEST["split"] == "dev"].sort_values("seg_id").reset_index(drop=True)

assert set(DEV["audio_filename"]) == set(SPLIT.loc[SPLIT["split"] == "dev", "audio_filename"]), \
    "manifest dev recordings and split.csv dev recordings disagree"
assert set(TRAIN["audio_filename"]).isdisjoint(set(DEV["audio_filename"])), \
    "a recording appears in both train and dev - the split is not grouped"

print("train segments : %5d   recordings %3d   hours %.2f"
      % (len(TRAIN), TRAIN["audio_filename"].nunique(), TRAIN["dur_s"].sum() / 3600))
print("dev   segments : %5d   recordings %3d   hours %.2f"
      % (len(DEV), DEV["audio_filename"].nunique(), DEV["dur_s"].sum() / 3600))
print("lexicon entries:", len(LEX))

# The fixed dev subset used for eval DURING training. Proportional per recording
# so no single speaker dominates, deterministic from SEED so every run and every
# ablation selects the same clips.
def stratified_subset(df, n, seed):
    if n >= len(df):
        return df.reset_index(drop=True)
    rng = np.random.RandomState(seed)
    parts = []
    groups = list(df.groupby("audio_filename", sort=True))
    quota = {g: max(1, int(round(n * len(sub) / len(df)))) for g, sub in groups}
    for g, sub in groups:
        k = min(quota[g], len(sub))
        idx = rng.choice(len(sub), size=k, replace=False)
        parts.append(sub.iloc[sorted(idx)])
    outdf = pd.concat(parts).sort_values("seg_id").reset_index(drop=True)
    if len(outdf) > n:
        keep = rng.choice(len(outdf), size=n, replace=False)
        outdf = outdf.iloc[sorted(keep)].reset_index(drop=True)
    return outdf

DEV_SUB = stratified_subset(DEV, EVAL_SUBSET_N, SEED)
print("dev subset for in-training eval: %d segments over %d recordings"
      % (len(DEV_SUB), DEV_SUB["audio_filename"].nunique()))

head("LOAD - OFFICIAL DEV (primary validation layer)")
ODEV = None
if ODEV_META is not None:
    ODEV = pd.read_csv(ODEV_META, sep="\t", dtype=str, keep_default_na=False)
    n_files = len(list(ODEV_CLIPS.glob("*.mp3")))
    names_meta = set(ODEV["audio_filename"])
    names_disk = set(p.name for p in ODEV_CLIPS.glob("*.mp3"))
    print("metadata rows : %d" % len(ODEV))
    print("clips on disk : %d" % n_files)
    print("names match   : %s  (meta-only %d, disk-only %d)"
          % (names_meta == names_disk, len(names_meta - names_disk), len(names_disk - names_meta)))
    assert names_meta == names_disk, "official dev metadata and clips do not line up one-to-one"
    # Same normalizer as every other reference in this project.
    ODEV["ref_official"] = ODEV["transcript"].map(scorer.official_normalize)
    ODEV["ref_folded"]   = ODEV["ref_official"].map(fold)
    print("language mix  :", ODEV["language"].value_counts().to_dict())
    print("conversations :", ODEV["convo_id"].value_counts().to_dict())
    print(ODEV[["audio_filename", "speaker", "language"]].head(3).to_string(index=False))
    _dia = sum(1 for t in ODEV["ref_official"] for w in t.split() if fold(w) != w)
    _tok = sum(len(t.split()) for t in ODEV["ref_official"])
    print("diacritic tokens in the official reference: %d / %d (%.2f%%)"
          % (_dia, _tok, 100.0 * _dia / max(_tok, 1)))
else:
    print("!!! indonesian_dev is not attached. The PRIMARY validation layer will be")
    print("!!! skipped, and Gate 2 cannot be decided on it. Attach the dataset.")

# Fixed subset for the in-training official curve. Seeded, sorted, so every run
# and every ablation watches the same clips. The full 372 are still used for the
# comparator and the final evaluation - this is only the progress curve.
ODEV_SUB = None
if ODEV is not None and OFFICIAL_EVAL_DURING_TRAINING:
    ODEV_SUB = (ODEV.copy() if OFFICIAL_EVAL_N >= len(ODEV)
                else ODEV.sample(n=OFFICIAL_EVAL_N, random_state=SEED)
                        .sort_values("audio_filename").reset_index(drop=True))
    print("official curve subset: %d of %d clips, %d conversations"
          % (len(ODEV_SUB), len(ODEV), ODEV_SUB["convo_id"].nunique()))

rec("data", n_train_segments=len(TRAIN), n_dev_segments=len(DEV),
    train_hours=round(float(TRAIN["dur_s"].sum()) / 3600, 3),
    dev_hours=round(float(DEV["dur_s"].sum()) / 3600, 3),
    n_dev_subset=len(DEV_SUB), n_lexicon_entries=len(LEX),
    n_official_dev=(len(ODEV) if ODEV is not None else 0),
    official_dev_available=bool(ODEV is not None))

## 4 — Training items: native segments, or glued to ~25 s

The official clips average **24.9 s**; Jember segments average **5.4 s**. Training
only on short pieces teaches a shape the test set does not have.

Notebook 02 established that Jember's annotation is a **contiguous partition** —
all 6,471 inter-segment gaps are exactly 0 — and the audio is stored per whole
recording with sample offsets. So gluing consecutive segments is just a wider
`[start_sample, end_sample]` window and a joined transcript. No new audio, no
resegmentation, nothing to re-verify.

`SEGMENT_MODE = "concat25"` is therefore an ablation that costs one line, and
the contiguity is asserted rather than assumed — if a gap is not zero, the group
is cut there instead of silently swallowing untranscribed audio.

**And it is not only a distribution fix.** Whisper's feature extractor pads every
clip to the same 30 s window, so a training step costs the same for a 5 s item as
for a 25 s one. Checked against the real manifest: native gives 5,541 items
averaging 5.4 s — about 82 % of the encoder's work is padding — while `concat25`
gives **1,455 items averaging 20.7 s**, same audio, same tokens, **~3.8× fewer
optimizer steps per epoch**. If GPU quota is the binding constraint, this is the
first ablation to run, not the last.

In [ ]:
# ============================================================
# CELL 4 - TRAINING ITEMS
# ============================================================
head("TRAINING ITEMS (%s)" % SEGMENT_MODE)
SR = 16000
TARGET_S = 25.0     # aim for the official dev's mean clip length
HARD_CAP_S = 29.0   # Whisper's encoder window is 30 s; leave headroom

def build_items(df, mode):
    # An item is (item_id, recording, start_sample, end_sample, text, dur_s).
    items = []
    if mode == "native":
        for r in df.itertuples():
            items.append(dict(item_id=r.seg_id, rec=r.audio_filename, path=r.path,
                              start=int(r.start_sample), end=int(r.end_sample),
                              text=r.text_folded, dur_s=float(r.dur_s), n_seg=1))
        return items, dict(n_gap_breaks=0, n_cap_breaks=0)
    n_gap_breaks = n_cap_breaks = 0
    for rec_id, g in df.groupby("audio_filename", sort=True):
        g = g.sort_values("start_sample")
        buf = []
        def flush(buf):
            if not buf:
                return None
            first, last = buf[0], buf[-1]
            return dict(item_id="%s_c%s" % (rec_id, first.seg_id.split("_")[-1]),
                        rec=rec_id, path=first.path,
                        start=int(first.start_sample), end=int(last.end_sample),
                        text=" ".join(x.text_folded for x in buf).strip(),
                        dur_s=(int(last.end_sample) - int(first.start_sample)) / float(SR),
                        n_seg=len(buf))
        for r in g.itertuples():
            if buf:
                gap = int(r.start_sample) - int(buf[-1].end_sample)
                span = (int(r.end_sample) - int(buf[0].start_sample)) / float(SR)
                if gap != 0:
                    n_gap_breaks += 1
                    it = flush(buf); items.append(it); buf = []
                elif span > HARD_CAP_S or (span > TARGET_S and len(buf) >= 1):
                    n_cap_breaks += 1
                    it = flush(buf); items.append(it); buf = []
            if (int(r.end_sample) - int(r.start_sample)) / float(SR) > HARD_CAP_S:
                # A single segment already longer than the cap cannot be glued to
                # anything; it goes through alone and gets truncated by the
                # feature extractor exactly as it would in "native" mode.
                if buf:
                    items.append(flush(buf)); buf = []
                items.append(dict(item_id=r.seg_id, rec=rec_id, path=r.path,
                                  start=int(r.start_sample), end=int(r.end_sample),
                                  text=r.text_folded,
                                  dur_s=(int(r.end_sample) - int(r.start_sample)) / float(SR),
                                  n_seg=1))
                continue
            buf.append(r)
        if buf:
            items.append(flush(buf))
    return items, dict(n_gap_breaks=n_gap_breaks, n_cap_breaks=n_cap_breaks)

TRAIN_ITEMS, ITEM_STATS = build_items(TRAIN, SEGMENT_MODE)
# Dev is ALWAYS native, in every run and every ablation, or the numbers stop
# being comparable across the project.
DEV_ITEMS, _     = build_items(DEV, "native")
DEV_SUB_ITEMS, _ = build_items(DEV_SUB, "native")

_d = np.array([it["dur_s"] for it in TRAIN_ITEMS])
print("train items : %d   mean %.2f s   median %.2f s   max %.2f s   total %.2f h"
      % (len(TRAIN_ITEMS), _d.mean(), np.median(_d), _d.max(), _d.sum() / 3600))
print("over 29 s   : %d" % int((_d > 29.0).sum()))
print("breaks      : gap %d, cap %d" % (ITEM_STATS["n_gap_breaks"], ITEM_STATS["n_cap_breaks"]))
print("dev items   : %d (native)   dev subset items: %d" % (len(DEV_ITEMS), len(DEV_SUB_ITEMS)))
if SEGMENT_MODE == "concat25" and ITEM_STATS["n_gap_breaks"] > 0:
    print("!! %d groups were cut on a NON-ZERO gap. NB 02 measured all 6,471 gaps as"
          % ITEM_STATS["n_gap_breaks"])
    print("!! exactly 0, so this is worth understanding before trusting the ablation.")

# Whisper's feature extractor pads EVERY clip to the same 30 s window, so a
# training step costs the same whether the item is 5 s or 25 s. Native items
# therefore spend about 82 % of the encoder on padding, and gluing them is not
# only a distribution fix - it is a ~3.8x cheaper epoch.
STEPS_PER_EPOCH = math.ceil(len(TRAIN_ITEMS) / (PER_DEVICE_BATCH * GRAD_ACCUM))
TOTAL_STEPS_EST = STEPS_PER_EPOCH * NUM_EPOCHS
print("")
print("effective batch %d -> %d steps/epoch, %d steps for %d epochs"
      % (PER_DEVICE_BATCH * GRAD_ACCUM, STEPS_PER_EPOCH, TOTAL_STEPS_EST, NUM_EPOCHS))
print("audio actually inside the 30 s window: %.1f%% (the rest is padding the"
      % (100.0 * _d.mean() / 30.0))
print("encoder pays for anyway - this is what SEGMENT_MODE='concat25' recovers)")

if EVAL_EVERY == "auto":
    EVAL_EVERY = max(25, int(round(TOTAL_STEPS_EST / 8.0 / 25.0)) * 25)
    print("EVAL_EVERY resolved to %d -> about %d eval points across the run"
          % (EVAL_EVERY, max(1, TOTAL_STEPS_EST // EVAL_EVERY)))
else:
    print("EVAL_EVERY fixed at %d -> about %d eval points across the run"
          % (EVAL_EVERY, max(1, TOTAL_STEPS_EST // EVAL_EVERY)))
if TOTAL_STEPS_EST // max(EVAL_EVERY, 1) < EARLY_STOP_PATIENCE + 1:
    print("!! fewer eval points than the early-stopping patience (%d) - early"
          % EARLY_STOP_PATIENCE)
    print("!! stopping can never fire, and only %d checkpoints will ever compete."
          % max(1, TOTAL_STEPS_EST // max(EVAL_EVERY, 1)))

rec("recipe", eval_every=EVAL_EVERY, steps_per_epoch=STEPS_PER_EPOCH,
    total_steps_est=TOTAL_STEPS_EST,
    pct_window_used=round(100.0 * float(_d.mean()) / 30.0, 1))
rec("items", segment_mode=SEGMENT_MODE, n_train_items=len(TRAIN_ITEMS),
    mean_dur_s=round(float(_d.mean()), 3), median_dur_s=round(float(np.median(_d)), 3),
    max_dur_s=round(float(_d.max()), 3), total_hours=round(float(_d.sum()) / 3600, 3),
    n_over_29s=int((_d > 29.0).sum()), n_dev_items=len(DEV_ITEMS),
    n_dev_subset_items=len(DEV_SUB_ITEMS), **ITEM_STATS)

## 5 — Audio into RAM, once

208 FLAC files, whole recordings, decoded once and held as float16 (~1.2 GB).
Every item is then a slice, not a decode. Reading each recording once instead of
seeking 5,600 times is also the difference between ~4 minutes and ~15 minutes of
startup.

In [ ]:
# ============================================================
# CELL 5 - AUDIO PRELOAD
# ============================================================
import soundfile as sf
head("AUDIO")

AUDIO = {}
def preload_audio(recs):
    t0 = time.time()
    total = 0
    for i, rec_id in enumerate(sorted(recs)):
        path = AUDIO_ROOT / ("recordings/%s.flac" % rec_id)
        y, sr = sf.read(str(path), dtype="float32")
        if sr != SR:
            raise ValueError("unexpected sample rate %d in %s" % (sr, path))
        if y.ndim > 1:
            y = y.mean(axis=1)
        AUDIO[rec_id] = y.astype(np.float16)
        total += y.shape[0]
        if (i + 1) % 50 == 0:
            print("  %3d/%d recordings, %.2f h, %.0f s elapsed"
                  % (i + 1, len(recs), total / SR / 3600, time.time() - t0))
    print("preloaded %d recordings, %.2f h, %.2f GB, in %.0f s"
          % (len(recs), total / SR / 3600, total * 2 / 1e9, time.time() - t0))
    return total

def item_audio(it):
    if PRELOAD_AUDIO and it["rec"] in AUDIO:
        return AUDIO[it["rec"]][it["start"]:it["end"]].astype(np.float32)
    y, sr = sf.read(str(AUDIO_ROOT / it["path"]), start=it["start"], stop=it["end"],
                    dtype="float32")
    if sr != SR:
        raise ValueError("unexpected sample rate %d for %s" % (sr, it["item_id"]))
    return y

NEEDED = set(it["rec"] for it in TRAIN_ITEMS) | set(it["rec"] for it in DEV_ITEMS)
if PRELOAD_AUDIO:
    if RUN_MODE == "sanity":
        # The sanity run only ever touches a handful of items; preloading all
        # 208 recordings would be most of its runtime and prove nothing.
        NEEDED = set(list(sorted(set(it["rec"] for it in TRAIN_ITEMS)))[:6]) | \
                 set(it["rec"] for it in DEV_SUB_ITEMS[:24])
        print("sanity run: preloading only %d recordings" % len(NEEDED))
    n_samples = preload_audio(NEEDED)
else:
    n_samples = 0
    print("PRELOAD_AUDIO is off - every item will seek into its FLAC on demand.")

# Prove the slicing is right before 5,600 of them go into a model: durations
# from the manifest and from the actual arrays must agree.
_probe = [TRAIN_ITEMS[0], TRAIN_ITEMS[len(TRAIN_ITEMS) // 2], TRAIN_ITEMS[-1]]
for it in _probe:
    if it["rec"] not in AUDIO and PRELOAD_AUDIO:
        continue
    a = item_audio(it)
    print("  %-14s %6.2f s expected, %6.2f s read, peak %.3f, text: %s"
          % (it["item_id"], it["dur_s"], len(a) / SR, float(np.abs(a).max()) if len(a) else 0.0,
             (it["text"][:60] + "...") if len(it["text"]) > 60 else it["text"]))
    assert abs(len(a) / SR - it["dur_s"]) < 0.02, "slice length disagrees with the manifest"

rec("audio", preloaded=bool(PRELOAD_AUDIO), n_recordings_loaded=len(AUDIO),
    ram_gb=round(n_samples * 2 / 1e9, 3))

## 6 — Processor, dataset, collator, model

The three classic Whisper fine-tuning traps, all handled here and all asserted
below rather than assumed:

1. **`-100` label masking.** Padding positions must be `-100`, or the model
   learns to emit pad tokens and the loss still looks reasonable.
2. **The special-token prefix.** The tokenizer emits
   `<|startoftranscript|><|jw|><|transcribe|><|notimestamps|>` in front of every
   target; `Trainer` shifts labels right and prepends `decoder_start_token_id`
   itself, so that first token has to be trimmed or it is fed twice.
3. **Language token.** The processor is built with `language="javanese"`, which
   is the tokenizer's name for `jw`. Getting it wrong does not raise — it
   silently trains against a different prefix than it decodes with.

In [ ]:
# ============================================================
# CELL 6 - PROCESSOR / DATASET / COLLATOR / MODEL
# ============================================================
from dataclasses import dataclass
from typing import Any, Dict, List
from torch.utils.data import Dataset as TorchDataset
from transformers import (WhisperProcessor, WhisperForConditionalGeneration,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          EarlyStoppingCallback, TrainerCallback)

head("PROCESSOR")
PROCESSOR = WhisperProcessor.from_pretrained(HF_MODEL, language=LANG_NAME, task="transcribe")
TOK = PROCESSOR.tokenizer
FEX = PROCESSOR.feature_extractor
N_MELS = FEX.feature_size
print("mel bins: %d   sampling rate: %d   chunk: %d s" % (N_MELS, FEX.sampling_rate, FEX.chunk_length))

# Prove the prefix is the one we think it is, before 8.5 hours of audio is
# labelled with it.
_probe_ids = TOK("halo rek", add_special_tokens=True).input_ids
_probe_toks = TOK.convert_ids_to_tokens(_probe_ids)
print("label prefix:", _probe_toks[:4])
assert any(t == "<|%s|>" % LANG_CODE for t in _probe_toks[:4]), \
    "the tokenizer prefix does not carry <|%s|> - language=%r produced %r" % (LANG_CODE, LANG_NAME, _probe_toks[:4])
MAX_LABEL_LEN = TOK.model_max_length if TOK.model_max_length and TOK.model_max_length < 1e4 else 448
print("max label length:", MAX_LABEL_LEN)


class WhisperItems(TorchDataset):
    # Tokenizes once at construction (cheap, and it lets over-long targets be
    # dropped before training rather than silently truncated); the mel transform
    # happens in the collator, batched.
    def __init__(self, items, tokenizer, max_label_len):
        self.items = []
        self.n_dropped = 0
        for it in items:
            ids = tokenizer(it["text"], add_special_tokens=True).input_ids
            if len(ids) > max_label_len:
                self.n_dropped += 1
                continue
            self.items.append(dict(it, labels=ids))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        it = self.items[i]
        return {"audio": item_audio(it), "labels": it["labels"], "item_id": it["item_id"]}


@dataclass
class WhisperCollator:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        batch = self.processor.feature_extractor(
            [f["audio"] for f in features], sampling_rate=SR, return_tensors="pt")
        label_batch = self.processor.tokenizer.pad(
            [{"input_ids": f["labels"]} for f in features], return_tensors="pt")
        # THE masking line. Anything that was padding becomes -100 and is
        # excluded from the loss.
        labels = label_batch["input_ids"].masked_fill(label_batch.attention_mask.ne(1), -100)
        # Trainer prepends decoder_start_token_id when it shifts the labels
        # right, so the tokenizer's own leading <|startoftranscript|> has to go.
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch


def apply_gen_config(model):
    # One decode configuration for everything this notebook measures: the
    # fine-tuned model, the zero-shot comparator, and every ablation.
    gc = model.generation_config
    gc.language = LANG_CODE
    gc.task = "transcribe"
    gc.num_beams = 1
    gc.do_sample = False
    # NOT temperature=0.0. Greedy decoding is do_sample=False; temperature is a
    # sampling-only knob, and a GenerationConfig that carries a non-default
    # temperature with do_sample=False fails strict validation - which
    # save_pretrained runs, so it surfaces at the FIRST CHECKPOINT SAVE rather
    # than here. Notebook 03 never hit it because it passed temperature as a
    # generate() argument, which is never saved. Same greedy decode either way.
    for attr, default in (("temperature", 1.0), ("top_p", 1.0), ("top_k", 50)):
        if getattr(gc, attr, None) is not None:
            setattr(gc, attr, default)
    gc.no_repeat_ngram_size = NO_REPEAT_NGRAM
    gc.max_length = GEN_MAX_LENGTH
    gc.use_cache = True          # gradient checkpointing turns the cache off for
                                 # the training forward pass; generation wants it
    try:
        # Legacy path. If it survives alongside language/task it silently wins,
        # and the model decodes with a prefix the labels were never built with.
        gc.forced_decoder_ids = None
    except Exception:
        pass
    # Trainer calls save_pretrained on every checkpoint, and save_pretrained
    # runs validate(strict=True). Running it here means an invalid config costs
    # one second at cell 6 instead of dying at the first save, twenty minutes in.
    try:
        gc.validate(strict=True)
    except TypeError:
        pass                      # older transformers: no strict= argument
    except ValueError as e:
        raise ValueError(
            "The generation config this notebook builds would be rejected when "
            "Trainer saves its first checkpoint, so it is rejected here instead. "
            "Fix apply_gen_config() in this cell.\n\n%s" % e)
    return model


def build_model(variant, for_training=True):
    dtype = torch.float32 if for_training else (torch.float16 if DEVICE == "cuda" else torch.float32)
    model = WhisperForConditionalGeneration.from_pretrained(HF_MODEL, torch_dtype=dtype)
    model.config.use_cache = not (for_training and GRAD_CHECKPOINT)
    if for_training and SPEC_AUGMENT:
        # The main regulariser on 8.5 hours of data.
        model.config.apply_spec_augment = True
        model.config.mask_time_prob = 0.05
        model.config.mask_time_length = 10
        model.config.mask_feature_prob = 0.05
        model.config.mask_feature_length = 10
    n_before = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if for_training and variant in ("freeze_encoder", "freeze_encoder_half"):
        enc = model.model.encoder
        layers = list(enc.layers)
        if variant == "freeze_encoder":
            for p in enc.parameters():
                p.requires_grad = False
        else:
            half = len(layers) // 2
            for p in enc.conv1.parameters(): p.requires_grad = False
            for p in enc.conv2.parameters(): p.requires_grad = False
            for lyr in layers[:half]:
                for p in lyr.parameters():
                    p.requires_grad = False
        print("frozen: %s (%d encoder layers total)" % (variant, len(layers)))
    if for_training and variant == "lora":
        from peft import LoraConfig, get_peft_model
        cfg = LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, bias="none",
                         target_modules=["q_proj", "v_proj"])
        model = get_peft_model(model, cfg)
        model.print_trainable_parameters()
    if for_training and GRAD_CHECKPOINT:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
    # NOT hasattr(model, "base_model"): that is a property every PreTrainedModel
    # carries, so it is true for a plain Whisper too and sends this down the PEFT
    # path. peft_config only exists on an actual PEFT wrapper.
    apply_gen_config(model.get_base_model() if hasattr(model, "peft_config") else model)
    n_after = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("params: %.1f M total, %.1f M trainable (%.1f%%)"
          % (sum(p.numel() for p in model.parameters()) / 1e6, n_after / 1e6,
             100.0 * n_after / max(n_before, 1)))
    return model


COLLATOR = WhisperCollator(PROCESSOR, PROCESSOR.tokenizer.convert_tokens_to_ids("<|startoftranscript|>"))
print("decoder_start_token_id used for trimming:", COLLATOR.decoder_start_token_id)

rec("model_setup", n_mels=N_MELS, max_label_len=int(MAX_LABEL_LEN),
    label_prefix=[str(t) for t in _probe_toks[:4]],
    decoder_start_token_id=int(COLLATOR.decoder_start_token_id))

## 7 — Decode and evaluation helpers

Two decode paths, because the two validation layers have different shapes:

- **Short (< 29.8 s)** — every Jember segment and 89.8 % of official clips.
  Batched, one 30 s encoder window.
- **Long** — the 10.2 % of official clips over Whisper's window. Primary path is
  transformers' sequential long-form generation (`truncation=False` +
  `return_timestamps=True`); if that raises, the fallback is a plain 28 s chunker
  whose texts are concatenated. Which path each clip took is recorded, because
  notebook 08 has to make the same choice for 2,118 test clips and would
  otherwise silently truncate about 216 of them.

In [ ]:
# ============================================================
# CELL 7 - DECODE + EVAL HELPERS
# ============================================================
LONG_S = 29.8
DECODE_PATHS = collections.Counter()

def _gen_kwargs():
    return dict(task="transcribe", language=LANG_CODE, num_beams=1, do_sample=False,
                no_repeat_ngram_size=NO_REPEAT_NGRAM, max_length=GEN_MAX_LENGTH,
                return_timestamps=False)

def transcribe_short(model, arrays):
    inputs = PROCESSOR.feature_extractor(arrays, sampling_rate=SR, return_tensors="pt")
    dtype = next(model.parameters()).dtype
    feats = inputs.input_features.to(DEVICE, dtype=dtype)
    with torch.no_grad():
        ids = model.generate(feats, **_gen_kwargs())
    DECODE_PATHS["short"] += len(arrays)
    return [t.strip() for t in PROCESSOR.batch_decode(ids, skip_special_tokens=True)]

def _join_overlapping(texts, max_overlap_words=8):
    # Chunks overlap by LONG_OVERLAP_S, so consecutive texts usually repeat a few
    # words at the seam. Drop the longest repeated run, longest match first.
    joined = ""
    for t in texts:
        t = (t or "").strip()
        if not t:
            continue
        if not joined:
            joined = t
            continue
        a, b = joined.split(), t.split()
        k = 0
        for n in range(min(max_overlap_words, len(a), len(b)), 0, -1):
            if [w.lower() for w in a[-n:]] == [w.lower() for w in b[:n]]:
                k = n
                break
        joined = " ".join(a + b[k:])
    return joined.strip()

def transcribe_chunked(model, y):
    # Fixed windows with an overlap, so a word straddling a boundary is seen
    # whole by at least one window; the duplicate is then trimmed at the seam.
    step = int((LONG_CHUNK_S - LONG_OVERLAP_S) * SR)
    win = int(LONG_CHUNK_S * SR)
    parts = [y[i:i + win] for i in range(0, len(y), step)]
    parts = [p for p in parts if len(p) > int(0.3 * SR)]
    if not parts:
        return ""
    texts = transcribe_short(model, parts)
    DECODE_PATHS["short"] -= len(parts)     # counted as one long clip, not N short
    DECODE_PATHS["chunk"] += 1
    DECODE_PATHS["chunk_windows"] += len(parts)
    return _join_overlapping(texts)

def transcribe_longform(model, y):
    dtype = next(model.parameters()).dtype
    inputs = PROCESSOR(y, sampling_rate=SR, return_tensors="pt",
                       truncation=False, padding="longest", return_attention_mask=True)
    kw = _gen_kwargs(); kw["return_timestamps"] = True
    with torch.no_grad():
        ids = model.generate(inputs.input_features.to(DEVICE, dtype=dtype),
                             attention_mask=inputs.attention_mask.to(DEVICE), **kw)
    DECODE_PATHS["long_form"] += 1
    return PROCESSOR.batch_decode(ids, skip_special_tokens=True)[0].strip()

def transcribe_long(model, y):
    # LONG_MODE is resolved once, for every model and both sides of the Gate 2
    # comparison. See the knob comment in cell 1 for why "auto" is not the
    # default: a per-clip fallback silently gave the zero-shot and fine-tuned
    # passes different decode algorithms on the same clips.
    if LONG_MODE == "chunk":
        return transcribe_chunked(model, y)
    if LONG_MODE == "long_form":
        return transcribe_longform(model, y)
    try:
        return transcribe_longform(model, y)
    except Exception as e:
        if DECODE_PATHS["long_fallback"] == 0:
            print("    [long-form unavailable: %r - chunking instead. NOTE: this run"
                  " is now mixing decode algorithms across clips.]" % (e,))
        DECODE_PATHS["long_fallback"] += 1
        return transcribe_chunked(model, y)

def decode_jember(model, items, batch_size=None, tag=""):
    bs = batch_size or EVAL_BATCH
    model.eval()
    texts, t0 = [], time.time()
    for i in range(0, len(items), bs):
        chunk = items[i:i + bs]
        texts.extend(transcribe_short(model, [item_audio(it) for it in chunk]))
        if tag and (i // bs) % 20 == 0 and i:
            print("    %s %d/%d  %.0f s" % (tag, i, len(items), time.time() - t0))
    print("  decoded %d Jember items in %.0f s (%.3f s/clip)"
          % (len(items), time.time() - t0, (time.time() - t0) / max(len(items), 1)))
    return texts

def load_official_audio(fname):
    p = ODEV_CLIPS / fname
    try:
        import librosa
        y, _ = librosa.load(str(p), sr=SR, mono=True)
        return y.astype(np.float32)
    except Exception:
        y, sr0 = sf.read(str(p), dtype="float32")
        if y.ndim > 1:
            y = y.mean(axis=1)
        if sr0 != SR:
            from scipy.signal import resample_poly
            g = math.gcd(int(sr0), SR)
            y = resample_poly(y, SR // g, int(sr0) // g)
        return y.astype(np.float32)

OFFICIAL_AUDIO = {}
def decode_official(model, df, tag="", reset_paths=True):
    # 372 clips, mean 24.9 s. Audio is cached across calls so the zero-shot
    # comparator and the fine-tuned model do not each pay the mp3 decode.
    # DECODE_PATHS is reset per pass: cumulative counters made it impossible to
    # see that the two Gate 2 passes had taken different routes. The in-training
    # curve passes reset_paths=False so it does not clear the pass it sits
    # inside; it runs after the comparator's counters are already recorded and
    # before the final pass resets them again.
    if reset_paths:
        DECODE_PATHS.clear()
    model.eval()
    hyps, t0 = [], time.time()
    short_buf, short_idx = [], []
    results = [None] * len(df)
    for i, r in enumerate(df.itertuples()):
        if r.audio_filename not in OFFICIAL_AUDIO:
            OFFICIAL_AUDIO[r.audio_filename] = load_official_audio(r.audio_filename)
        y = OFFICIAL_AUDIO[r.audio_filename]
        if len(y) / SR > LONG_S:
            results[i] = transcribe_long(model, y)
        else:
            short_buf.append(y); short_idx.append(i)
            if len(short_buf) == EVAL_BATCH:
                for j, t in zip(short_idx, transcribe_short(model, short_buf)):
                    results[j] = t
                short_buf, short_idx = [], []
        if tag and (i + 1) % 100 == 0:
            print("    %s %d/%d  %.0f s" % (tag, i + 1, len(df), time.time() - t0))
    if short_buf:
        for j, t in zip(short_idx, transcribe_short(model, short_buf)):
            results[j] = t
    print("  decoded %d official clips in %.0f s (%.2f s/clip)   paths: %s"
          % (len(df), time.time() - t0, (time.time() - t0) / max(len(df), 1), dict(DECODE_PATHS)))
    return [r if r is not None else "" for r in results]

def score_jember(items, hyps, prefix=""):
    # Two reference spaces. text_folded is primary (the training target and the
    # official dev's orthography); text_official is carried for continuity with
    # notebook 03, which had no folded-space guarded number.
    ids = [it["item_id"] for it in items]
    # This addresses the manifest by seg_id, which only works for native items -
    # a concatenated item carries a synthesised id. Dev is native in every mode,
    # so this is an invariant, not a limitation; state it rather than letting a
    # future change surface as a KeyError from inside pandas.
    assert all(it["n_seg"] == 1 for it in items), \
        "score_jember was handed concatenated items; dev must stay native"
    M = MANIFEST.set_index("seg_id")
    ref_folded   = [M.at[i, "text_folded"] for i in ids]
    ref_official = [M.at[i, "text_official"] for i in ids]
    d = {}
    d.update(wer_pack(ref_folded, [fold(h) for h in hyps], LEX, prefix="folded_"))
    d.update(wer_pack(ref_official, hyps, LEX, prefix="official_"))
    return d, ref_folded, ref_official

def score_official(df, hyps):
    d = {}
    d.update(wer_pack(list(df["ref_official"]), hyps, LEX, prefix="official_"))
    d.update(wer_pack(list(df["ref_folded"]), [fold(h) for h in hyps], LEX, prefix="folded_"))
    return d

def compute_metrics_factory(ref_texts):
    # Trainer hands back generated ids; the reference list is fixed and in the
    # same order as the eval dataset, so it is closed over rather than decoded
    # back out of the -100-masked labels.
    def compute_metrics(pred):
        ids = pred.predictions
        ids = np.where(ids != -100, ids, TOK.pad_token_id)
        hyps = [t.strip() for t in TOK.batch_decode(ids, skip_special_tokens=True)]
        w = scorer.wer([fold(r) for r in ref_texts[:len(hyps)]], [fold(h) for h in hyps])
        n_ref = sum(len(scorer.tokenize(r)) for r in ref_texts[:len(hyps)])
        n_hyp = sum(len(scorer.tokenize(h)) for h in hyps)
        return {"wer": round(w["wer"], 5),
                "ins_rate": round(w["ins_rate"], 5),
                "del_rate": round(w["del_rate"], 5),
                "sub_rate": round(w["sub_rate"], 5),
                "len_ratio": round(n_hyp / max(n_ref, 1), 4)}
    return compute_metrics

print("decode settings:", _gen_kwargs())
print("long-form threshold: %.1f s" % LONG_S)

## 8 — HuggingFace: private repo, two branches

Checkpoints never go into `findings/` and are never downloaded — the brief is
explicit, and a 12 h session that dies takes `/kaggle/working` with it.

- **`main`** — the deliverable: best weights + processor, ~1.6 GB.
- **`training-state`** — a *resumable* checkpoint: weights **plus** optimizer and
  scheduler state, several GB. This is the branch a killed session comes back
  to, and the one the cross-session resume proof reads.

The token comes from Kaggle Secrets under the name `HF_TOKEN` (Add-ons →
Secrets). Without it the notebook still trains and still measures everything —
it just cannot satisfy the last Gate 2 criterion.

In [ ]:
# ============================================================
# CELL 8 - HUGGINGFACE HUB
# ============================================================
head("HUGGINGFACE")
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
    if not HF_TOKEN:
        print("no Kaggle secret named HF_TOKEN (%r) and no env var either" % (e,))

HF_ENABLED = False
HF_REPO = None
HF_USER = None
STATE_BRANCH = "training-state"
if HF_TOKEN:
    try:
        from huggingface_hub import HfApi
        API = HfApi(token=HF_TOKEN)
        HF_USER = API.whoami()["name"]
        HF_REPO = "%s/mozilla-jv-%s" % (HF_USER, RUN_ID)
        API.create_repo(HF_REPO, private=True, exist_ok=True, repo_type="model")
        try:
            API.create_branch(HF_REPO, branch=STATE_BRANCH, exist_ok=True)
        except Exception as e:
            print("could not create the %s branch: %r" % (STATE_BRANCH, e))
        HF_ENABLED = True
        print("hf user : %s" % HF_USER)
        print("hf repo : %s  (private)" % HF_REPO)
        print("branches: main = best weights, %s = resumable checkpoint" % STATE_BRANCH)
    except Exception as e:
        print("!!! HF setup failed: %r" % (e,))
if not HF_ENABLED:
    print("!!! Hub disabled. Training will run, but nothing is durable and Gate 2's")
    print("!!! last criterion (best checkpoint on HF with a recorded revision)")
    print("!!! cannot pass. Add-ons -> Secrets -> HF_TOKEN, then re-run this cell.")

# The sanity suite runs under RUN_ID "<id>-sanity" and therefore its own repo.
# The training run is a different RUN_ID and a different repo, so it has to be
# told where the resume proof lives, or Gate 2 criterion 4 fails while staring
# straight at the evidence.
SANITY_REPO = ("%s/mozilla-jv-%s-sanity" % (HF_USER, NOTEBOOK_ID)) if HF_USER else None
# Same variant and segment mode, without the recipe note - a recipe variant can
# borrow its comparator, since the comparator depends only on decode settings.
BASE_REPO = ("%s/mozilla-jv-%s" % (HF_USER, BASE_RUN_ID)) if HF_USER else None

def hf_fetch_json(name, repos=None):
    # Tries this run's repo first, then the sanity repo. Returns None rather
    # than raising: a missing artifact is a finding, not a crash.
    if not HF_ENABLED:
        return None
    from huggingface_hub import hf_hub_download
    for repo in (repos or [HF_REPO, SANITY_REPO]):
        if not repo:
            continue
        try:
            p = hf_hub_download(repo, name, token=HF_TOKEN)
            print("  fetched %s from %s" % (name, repo))
            return json.loads(Path(p).read_text(encoding="utf-8"))
        except Exception:
            continue
    return None

def hf_push_folder(folder, branch, message):
    if not HF_ENABLED:
        return None
    t0 = time.time()
    try:
        info = API.upload_folder(folder_path=str(folder), repo_id=HF_REPO,
                                 repo_type="model", revision=branch,
                                 commit_message=message)
        sha = getattr(info, "oid", None) or getattr(info, "commit_id", None) or str(info)
        size = sum(p.stat().st_size for p in Path(folder).rglob("*") if p.is_file())
        print("  pushed %s -> %s@%s  (%.2f GB, %.0f s)  rev %s"
              % (Path(folder).name, HF_REPO, branch, size / 1e9, time.time() - t0, sha))
        return str(sha)
    except Exception as e:
        print("  !! push failed (training is unaffected): %r" % (e,))
        return None

def hf_pull_branch(branch, dest):
    if not HF_ENABLED:
        return None
    try:
        from huggingface_hub import snapshot_download
        p = snapshot_download(repo_id=HF_REPO, revision=branch, local_dir=str(dest),
                              token=HF_TOKEN)
        print("  pulled %s@%s -> %s" % (HF_REPO, branch, p))
        return Path(p)
    except Exception as e:
        print("  !! pull failed: %r" % (e,))
        return None

def hf_upload_file(path, branch="main", name=None):
    if not HF_ENABLED:
        return None
    try:
        API.upload_file(path_or_fileobj=str(path), path_in_repo=(name or Path(path).name),
                        repo_id=HF_REPO, repo_type="model", revision=branch)
        return True
    except Exception as e:
        print("  !! upload of %s failed: %r" % (path, e))
        return None


class PushStateCallback(TrainerCallback):
    # A resumable checkpoint carries optimizer state, so it is several GB.
    # Pushing every save would spend more wall-clock uploading than training;
    # every PUSH_EVERY_N_SAVES saves keeps the worst case bounded to that many
    # eval intervals of lost work.
    def __init__(self):
        self.n_saves = 0
        self.last_rev = None
    def on_save(self, args, state, control, **kw):
        self.n_saves += 1
        if not HF_ENABLED or (self.n_saves % PUSH_EVERY_N_SAVES) != 0:
            return
        ck = Path(args.output_dir) / ("checkpoint-%d" % state.global_step)
        if ck.is_dir():
            self.last_rev = hf_push_folder(ck, STATE_BRANCH,
                                           "training state @ step %d" % state.global_step)

rec("hub", enabled=HF_ENABLED, repo=HF_REPO, user=HF_USER, state_branch=STATE_BRANCH,
    push_every_n_saves=PUSH_EVERY_N_SAVES)

## 9 — Trainer factory, optimizer choice, speed probe

`Seq2SeqTrainingArguments` has renamed arguments between transformers versions
(`evaluation_strategy` → `eval_strategy` is the recent one). A rename should not
kill a run twenty minutes after the model finished loading, so the arguments are
filtered against the actual signature and anything unknown is reported and
dropped rather than raised.

**Optimizer.** Full fine-tuning of turbo in fp32 master weights needs ~3.2 GB of
gradients and ~6.5 GB of AdamW state on top of 3.2 GB of weights — 12.9 GB
before a single activation, on a 16 GB card. `adamw_bnb_8bit` cuts the optimizer
state to ~1.6 GB; `adafactor` cuts it to almost nothing. Whichever one is
actually used is recorded, because it is not free of consequence and an ablation
compared against a baseline that used a different optimizer is not an ablation.

**Speed probe.** Forward + backward only, gradients zeroed and no optimizer step,
so it measures the dominant cost without moving a single weight. The projection
adds 10 % for the optimizer step and the measured generation cost for evals.

In [ ]:
# ============================================================
# CELL 9 - TRAINER FACTORY + SPEED PROBE
# ============================================================
head("TRAINER FACTORY")

def pick_optim():
    if OPTIM != "auto":
        return OPTIM
    try:
        import bitsandbytes  # noqa: F401
        return "adamw_bnb_8bit"
    except Exception:
        return "adafactor"

OPTIM_USED = pick_optim()
print("optimizer: %s" % OPTIM_USED)
if OPTIM_USED == "adafactor":
    print("  (bitsandbytes not importable - adafactor keeps the optimizer state")
    print("   small enough for a 16 GB card, at slightly different dynamics)")

_TA_ALIASES = {"eval_strategy": ["eval_strategy", "evaluation_strategy"],
               "evaluation_strategy": ["eval_strategy", "evaluation_strategy"]}

def make_training_args(**kw):
    sig = set(inspect.signature(Seq2SeqTrainingArguments.__init__).parameters)
    fixed, dropped = {}, []
    for k, v in kw.items():
        if k in sig:
            fixed[k] = v
            continue
        alt = next((a for a in _TA_ALIASES.get(k, []) if a in sig), None)
        if alt:
            fixed[alt] = v
        else:
            dropped.append(k)
    if dropped:
        print("  [training args] not supported by transformers %s, dropped: %s"
              % (transformers.__version__, dropped))
    return Seq2SeqTrainingArguments(**fixed)

def base_targs(output_dir, **over):
    kw = dict(
        output_dir=str(output_dir),
        per_device_train_batch_size=PER_DEVICE_BATCH,
        per_device_eval_batch_size=EVAL_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="linear",
        num_train_epochs=NUM_EPOCHS,
        fp16=(DEVICE == "cuda"),
        gradient_checkpointing=GRAD_CHECKPOINT,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        eval_strategy="steps",
        eval_steps=EVAL_EVERY,
        save_strategy="steps",
        save_steps=EVAL_EVERY,
        logging_steps=25,
        save_total_limit=SAVE_TOTAL_LIMIT,
        save_safetensors=True,
        predict_with_generate=True,
        generation_max_length=GEN_MAX_LENGTH,
        load_best_model_at_end=True,
        metric_for_best_model="wer",     # dev WER, never loss (brief)
        greater_is_better=False,
        remove_unused_columns=False,
        label_names=["labels"],
        dataloader_num_workers=0,        # audio is already in RAM; workers would
                                         # only fork a 1.2 GB address space
        optim=OPTIM_USED,
        seed=SEED,
        data_seed=SEED,
        report_to=[],
        disable_tqdm=False,
    )
    kw.update(over)
    return make_training_args(**kw)

def build_trainer(model, train_ds, eval_ds, eval_refs, targs, callbacks=None):
    return Seq2SeqTrainer(
        model=model, args=targs, train_dataset=train_ds, eval_dataset=eval_ds,
        data_collator=COLLATOR, compute_metrics=compute_metrics_factory(eval_refs),
        processing_class=PROCESSOR, callbacks=(callbacks or []))

# processing_class= is the transformers>=4.46 name for what used to be
# tokenizer=. Fall back automatically rather than failing on an old image.
if "processing_class" not in set(inspect.signature(Seq2SeqTrainer.__init__).parameters):
    def build_trainer(model, train_ds, eval_ds, eval_refs, targs, callbacks=None):  # noqa: F811
        return Seq2SeqTrainer(
            model=model, args=targs, train_dataset=train_ds, eval_dataset=eval_ds,
            data_collator=COLLATOR, compute_metrics=compute_metrics_factory(eval_refs),
            tokenizer=PROCESSOR.feature_extractor, callbacks=(callbacks or []))
    print("  [trainer] this transformers still wants tokenizer=, not processing_class=")


class OfficialEvalCallback(TrainerCallback):
    # The official dev, measured DURING training instead of only at the end.
    # 30 Aug: Jember dev fell 40 points while the official layer did not move,
    # and because the official layer had exactly one measurement there was no
    # way to tell whether it had peaked early and then regressed. This makes
    # that visible. It never selects a checkpoint - see the knob comment in
    # cell 1 for why the official 372 stay a validation set, not a selector.
    def __init__(self, df):
        self.df = df
        self.rows = []
        self.disabled = df is None

    def on_evaluate(self, args, state, control, model=None, **kw):
        if self.disabled or model is None:
            return
        t0 = time.time()
        try:
            was_training = model.training
            model.eval()
            with torch.no_grad():
                hyps = decode_official(model, self.df, reset_paths=False)
            m = score_official(self.df, hyps)
            row = dict(step=int(state.global_step), epoch=round(float(state.epoch or 0), 3),
                       official_wer=m["official_wer"], official_wer_jv=m["official_wer_jv"],
                       official_wer_id=m["official_wer_id"],
                       official_jv_to_id=m["official_jv_to_id_sub_pct"],
                       official_len_ratio=m["official_len_ratio"],
                       seconds=round(time.time() - t0, 1))
            self.rows.append(row)
            print("  [OFFICIAL %d clips] step %-5d WER %.4f   jv %.4f  id %.4f   jv_to_id %.2f%%   %.0f s"
                  % (len(self.df), row["step"], row["official_wer"], row["official_wer_jv"],
                     row["official_wer_id"], row["official_jv_to_id"], row["seconds"]))
            if was_training:
                model.train()
        except Exception as e:
            # Never let a progress measurement kill a four-hour training run.
            self.disabled = True
            print("  [official curve failed, disabled for the rest of the run: %r]" % (e,))


def speed_probe(model, ds, n_micro=8, n_gen=16):
    # Forward+backward timing with the gradients thrown away: the weights are
    # never updated, so this can run on the model that is about to train.
    model.train()
    batches = [COLLATOR([ds[i] for i in range(j * PER_DEVICE_BATCH, (j + 1) * PER_DEVICE_BATCH)])
               for j in range(n_micro + 2) if (j + 1) * PER_DEVICE_BATCH <= len(ds)]
    if len(batches) < 3:
        batches = batches * 3
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
    except Exception:
        scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))
    times = []
    for i, b in enumerate(batches):
        b = {k: v.to(DEVICE) for k, v in b.items()}
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        with torch.autocast(device_type=("cuda" if DEVICE == "cuda" else "cpu"),
                            dtype=torch.float16, enabled=(DEVICE == "cuda")):
            loss = model(**b).loss
        scaler.scale(loss).backward()
        model.zero_grad(set_to_none=True)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        dt = time.time() - t0
        if i >= 2:            # first two pay cuDNN autotune, not model cost
            times.append(dt)
    peak = (torch.cuda.max_memory_allocated() / 1e9) if DEVICE == "cuda" else 0.0
    sec_micro = float(np.median(times)) if times else float("nan")

    # Generation cost, measured the same way the evals will pay it.
    model.eval()
    gen_items = [ds.items[i] for i in range(min(n_gen, len(ds)))]
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        for i in range(0, len(gen_items), EVAL_BATCH):
            transcribe_short(model, [item_audio(it) for it in gen_items[i:i + EVAL_BATCH]])
    sec_gen = (time.time() - t0) / max(len(gen_items), 1)
    model.train()
    print("  fwd+bwd  : %.3f s per micro-batch of %d  (peak VRAM %.1f GB)"
          % (sec_micro, PER_DEVICE_BATCH, peak))
    print("  generate : %.3f s per clip at batch %d" % (sec_gen, EVAL_BATCH))
    return dict(sec_per_microbatch=round(sec_micro, 4), sec_per_gen_clip=round(sec_gen, 4),
                peak_vram_gb=round(peak, 2))

def project_wallclock(n_items, probe):
    steps = math.ceil(n_items / (PER_DEVICE_BATCH * GRAD_ACCUM)) * NUM_EPOCHS
    train_s = steps * GRAD_ACCUM * probe["sec_per_microbatch"] * 1.10  # +10% optimizer step
    n_evals = max(1, steps // max(EVAL_EVERY, 1))
    eval_s = n_evals * EVAL_SUBSET_N * probe["sec_per_gen_clip"]
    # The 29 Aug projection stopped here and so understated the run: cells 11 and
    # 13 each decode the FULL Jember dev plus all 372 official clips, and the
    # official ones average 24.9 s, so ~10 % of them cost two encoder windows.
    n_off = len(ODEV) if ODEV is not None else 0
    per_pass = len(DEV_ITEMS) + n_off * 1.5
    extra_s = 2 * per_pass * probe["sec_per_gen_clip"]
    # The official curve: n_evals passes over ODEV_SUB, ~1.15 windows per clip
    # because ~10 % of the official clips run past the 30 s encoder window.
    curve_s = (n_evals * len(ODEV_SUB) * 1.15 * probe["sec_per_gen_clip"]) \
        if (ODEV_SUB is not None and OFFICIAL_EVAL_DURING_TRAINING) else 0.0
    eval_s += curve_s
    total_h = (train_s + eval_s + extra_s) / 3600.0
    print("  projected: %d optimizer steps, %d in-training evals" % (steps, n_evals))
    print("  projected: %.2f h train + %.2f h in-training eval (of which %.2f h is"
          % (train_s / 3600, eval_s / 3600, curve_s / 3600))
    print("             the official curve) + %.2f h comparator" % (extra_s / 3600,))
    print("             and final eval over %d Jember + %d official clips"
          % (len(DEV_ITEMS), n_off))
    print("  projected: %.2f h TOTAL   (budget %.1f h)" % (total_h, MAX_TRAIN_HOURS))
    return dict(n_steps=steps, n_evals=n_evals, train_hours=round(train_s / 3600, 3),
                eval_hours=round(eval_s / 3600, 3),
                comparator_and_final_eval_hours=round(extra_s / 3600, 3),
                total_hours=round(total_h, 3))

rec("optim", requested=OPTIM, used=OPTIM_USED)

## 10 — Sanity suite (`RUN_MODE = "sanity"`)

Three things, in order, none of them skippable, straight from the brief:

1. **Overfit 32 segments.** Train until the model reproduces its own training
   text. If this fails, the bug is in the collator or the label masking — not in
   the hyperparameters, and no amount of GPU time will fix it.
2. **Checkpoint-resume.** Train, save, throw the trainer and the model away,
   rebuild from nothing, resume, keep training. Optimizer state, scheduler
   position and step counter all have to come back. Kaggle sessions die at ~12 h,
   so this is a requirement, not a nicety.
3. **Steps per second on the real configuration**, then extrapolate. If the
   projection exceeds `MAX_TRAIN_HOURS`, shrink *before* starting.

Step 2 here is the in-process half. The **cross-session** half is a second sanity
run in a fresh session with `RESUME_FROM_HF = True`, which pulls the pushed state
back down and continues from it — that is the one that proves the session-death
path, and cell 16 will not mark it proven until both halves have been seen.

In [ ]:
# ============================================================
# CELL 10 - SANITY SUITE
# ============================================================
SANITY = {}
RESUME_TEST_STEPS = 30      # the brief says 100; what is being tested is state
                            # restoration, which 30 steps demonstrate just as
                            # conclusively at a fifth of the GPU cost

if RUN_MODE != "sanity":
    print("RUN_MODE = %r - skipping the sanity suite." % RUN_MODE)
    print("It must have been run once already; cell 16 checks for its report.")
else:
    # ---------- shared tiny data ----------
    tiny_items = TRAIN_ITEMS[:32] if not PRELOAD_AUDIO else \
                 [it for it in TRAIN_ITEMS if it["rec"] in AUDIO][:32]
    tiny_dev = [it for it in DEV_SUB_ITEMS if (not PRELOAD_AUDIO or it["rec"] in AUDIO)][:24]
    TINY = WhisperItems(tiny_items, TOK, MAX_LABEL_LEN)
    TINY_DEV = WhisperItems(tiny_dev, TOK, MAX_LABEL_LEN)
    refs_tiny = [it["text"] for it in TINY.items]
    print("tiny train %d items, tiny dev %d items (dropped over-long targets: %d/%d)"
          % (len(TINY), len(TINY_DEV), TINY.n_dropped, TINY_DEV.n_dropped))

    STAGES = ("overfit", "resume", "cross_session", "speed") if SANITY_STAGES == "all" \
             else tuple(SANITY_STAGES)
    print("stages this session: %s" % (list(STAGES),))

    # Carry forward what an earlier sanity session already proved, so a session
    # that runs one stage still ends with a complete report. Without this, run 2
    # would overwrite run 1's overfit verdict with nothing, and cell 16 would
    # call resume unproven while looking straight at the proof.
    _prev = None
    if out("sanity_report.json").exists():
        _prev = json.loads(out("sanity_report.json").read_text(encoding="utf-8"))
    else:
        _prev = hf_fetch_json("sanity_report.json")
        if _prev is None:
            print("no earlier sanity report to carry forward")
    if _prev:
        SANITY.update(_prev)
        print("carried forward stages: %s"
              % [k for k in ("collator", "overfit", "resume", "speed") if k in SANITY])

    # ---------- collator inspection: the -100 rule, seen ----------
    head("SANITY A0 - COLLATOR")
    _b = COLLATOR([TINY[i] for i in range(min(4, len(TINY)))])
    _lab = _b["labels"]
    print("input_features:", tuple(_b["input_features"].shape), _b["input_features"].dtype)
    print("labels        :", tuple(_lab.shape))
    print("first row     :", TOK.convert_ids_to_tokens([int(x) for x in _lab[0][:6]]))
    n_masked = int((_lab == -100).sum())
    print("masked positions: %d of %d (%.1f%%)" % (n_masked, _lab.numel(), 100.0 * n_masked / _lab.numel()))
    assert (_lab[:, 0] != COLLATOR.decoder_start_token_id).all(), \
        "the <|startoftranscript|> prefix was NOT trimmed - it would be fed twice"
    assert n_masked >= 0 and (_lab.max() < len(TOK)), "label ids out of vocabulary range"
    SANITY["collator"] = dict(feat_shape=list(_b["input_features"].shape),
                              label_shape=list(_lab.shape), pct_masked=round(100.0 * n_masked / _lab.numel(), 2),
                              prefix_trimmed=True)

    if "overfit" in STAGES:
        # ---------- A. overfit 32 ----------
        head("SANITY A - OVERFIT 32 SEGMENTS")
        t0 = time.time()
        m = build_model(TRAIN_VARIANT, for_training=True).to(DEVICE)
        print("model loaded in %.0f s" % (time.time() - t0))
        refs_tiny = [it["text"] for it in TINY.items]
        targs = base_targs(CKPT_DIR / "overfit", num_train_epochs=1, max_steps=100,
                           learning_rate=5e-5, warmup_steps=5, warmup_ratio=0.0,
                           eval_strategy="no", save_strategy="no",
                           load_best_model_at_end=False, logging_steps=10)
        tr = build_trainer(m, TINY, TINY, refs_tiny, targs)
        tr.train()
        _losses = [h["loss"] for h in tr.state.log_history if "loss" in h]
        loss_end = float(_losses[-1]) if _losses else float("nan")
        print("loss trace:", [round(x, 3) for x in _losses])
        hyp_tiny = decode_jember(m, TINY.items, tag="overfit")
        w_tiny = scorer.wer([fold(r) for r in refs_tiny], [fold(h) for h in hyp_tiny])["wer"]
        print("final training loss: %.4f" % loss_end)
        print("WER of the model against its OWN 32 training targets: %.4f" % w_tiny)
        print("  ref: %s" % refs_tiny[0][:90])
        print("  hyp: %s" % hyp_tiny[0][:90])
        overfit_ok = (w_tiny < 0.30 and loss_end < 1.0)
        print(">>> %s" % ("OVERFIT OK - labels and masking are wired correctly."
                          if overfit_ok else
                          "OVERFIT FAILED - the bug is in the collator or the label masking, "
                          "not in the hyperparameters. Do not start a long run."))
        SANITY["overfit"] = dict(n_items=len(TINY), steps=100, final_loss=round(loss_end, 4),
                                 wer_on_own_targets=round(float(w_tiny), 4), passed=bool(overfit_ok))
        del tr, m
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    if "resume" in STAGES:
        # ---------- B. checkpoint-resume, in process ----------
        head("SANITY B - CHECKPOINT / RESUME")
        rdir = CKPT_DIR / "resume"
        shutil.rmtree(rdir, ignore_errors=True)
        m = build_model(TRAIN_VARIANT, for_training=True).to(DEVICE)
        targs = base_targs(rdir, num_train_epochs=1, max_steps=RESUME_TEST_STEPS,
                           eval_strategy="no", save_strategy="steps",
                           save_steps=RESUME_TEST_STEPS, load_best_model_at_end=False,
                           logging_steps=5, save_total_limit=4)
        tr = build_trainer(m, TINY, TINY, refs_tiny, targs)
        tr.train()
        ck = rdir / ("checkpoint-%d" % RESUME_TEST_STEPS)
        files = sorted(p.name for p in ck.iterdir()) if ck.is_dir() else []
        has_opt = any(f.startswith("optimizer") for f in files)
        has_sched = any(f.startswith("scheduler") for f in files)
        lr_before = [h.get("learning_rate") for h in tr.state.log_history if "learning_rate" in h]
        print("checkpoint files:", files)
        print("optimizer state present: %s   scheduler state present: %s" % (has_opt, has_sched))

        # Everything is thrown away here on purpose. A resume that works only
        # because the old objects are still in memory proves nothing.
        del tr, m
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

        m2 = build_model(TRAIN_VARIANT, for_training=True).to(DEVICE)
        targs2 = base_targs(rdir, num_train_epochs=1, max_steps=RESUME_TEST_STEPS * 2,
                            eval_strategy="no", save_strategy="steps",
                            save_steps=RESUME_TEST_STEPS, load_best_model_at_end=False,
                            logging_steps=5, save_total_limit=4)
        tr2 = build_trainer(m2, TINY, TINY, refs_tiny, targs2)
        tr2.train(resume_from_checkpoint=str(ck))
        step_after = int(tr2.state.global_step)
        lr_after = [h.get("learning_rate") for h in tr2.state.log_history if "learning_rate" in h]
        resume_ok = (has_opt and has_sched and step_after == RESUME_TEST_STEPS * 2)
        print("global_step after resume: %d (expected %d)" % (step_after, RESUME_TEST_STEPS * 2))
        print("lr before resume, last 3: %s" % [round(x, 10) for x in lr_before[-3:]])
        print("lr after  resume, first 3: %s" % [round(x, 10) for x in lr_after[:3]])
        print(">>> %s" % ("IN-PROCESS RESUME OK." if resume_ok else "RESUME FAILED - a dead session would cost the whole run."))

        state_rev = None
        if HF_ENABLED:
            ck2 = rdir / ("checkpoint-%d" % (RESUME_TEST_STEPS * 2))
            push_dir = ck2 if ck2.is_dir() else ck
            print("pushing training state so a FRESH session can resume from it...")
            state_rev = hf_push_folder(push_dir, STATE_BRANCH, "sanity training state")
        del tr2, m2
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        SANITY["resume"] = dict(steps_first=RESUME_TEST_STEPS, steps_after=step_after,
                                optimizer_state=bool(has_opt), scheduler_state=bool(has_sched),
                                in_process_passed=bool(resume_ok),
                                # never claimed by this stage - only stage B2,
                                # in a different session, can prove it
                                cross_session_passed=bool(
                                    (SANITY.get("resume") or {}).get("cross_session_passed", False)),
                                pushed_revision=state_rev)

    # ---------- B2. cross-session resume, when asked for ----------
    if "cross_session" in STAGES and RESUME_FROM_HF:
        head("SANITY B2 - CROSS-SESSION RESUME (from HF)")
        pulled = hf_pull_branch(STATE_BRANCH, CKPT_DIR / "pulled")
        if pulled is None:
            print("!!! nothing pulled - cross-session resume NOT proven")
        else:
            st = json.loads((pulled / "trainer_state.json").read_text())
            start_step = int(st.get("global_step", 0))
            print("pulled checkpoint sits at global_step %d" % start_step)
            m3 = build_model(TRAIN_VARIANT, for_training=True).to(DEVICE)
            targs3 = base_targs(CKPT_DIR / "xsession", num_train_epochs=1,
                                max_steps=start_step + RESUME_TEST_STEPS,
                                eval_strategy="no", save_strategy="no",
                                load_best_model_at_end=False, logging_steps=5)
            tr3 = build_trainer(m3, TINY, TINY, refs_tiny, targs3)
            tr3.train(resume_from_checkpoint=str(pulled))
            xs_ok = int(tr3.state.global_step) == start_step + RESUME_TEST_STEPS
            print("global_step now %d (expected %d)" % (tr3.state.global_step, start_step + RESUME_TEST_STEPS))
            print(">>> %s" % ("CROSS-SESSION RESUME OK - a 12 h session death is survivable."
                             if xs_ok else "CROSS-SESSION RESUME FAILED."))
            SANITY.setdefault("resume", {})["cross_session_passed"] = bool(xs_ok)
            SANITY["resume"]["cross_session_start_step"] = start_step
            del tr3, m3
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

    if "speed" in STAGES:
        # ---------- C. speed probe + projection ----------
        head("SANITY C - STEPS PER SECOND, AND THE PROJECTION")
        m4 = build_model(TRAIN_VARIANT, for_training=True).to(DEVICE)
        probe = speed_probe(m4, TINY)
        proj = project_wallclock(len(TRAIN_ITEMS), probe)
        fits = proj["total_hours"] <= MAX_TRAIN_HOURS
        print(">>> %s" % ("PROJECTION FITS the budget." if fits else
                          "PROJECTION EXCEEDS %.1f h - shrink NUM_EPOCHS, raise GRAD_ACCUM, "
                          "or switch TRAIN_VARIANT to freeze_encoder BEFORE starting." % MAX_TRAIN_HOURS))
        if probe["peak_vram_gb"] > 13.0:
            print("!! peak VRAM %.1f GB on a 16 GB card - drop PER_DEVICE_BATCH to 2 and"
                  % probe["peak_vram_gb"])
            print("!! raise GRAD_ACCUM to 8 to keep the effective batch at 16.")
        SANITY["speed"] = dict(probe, **proj)
        SANITY["speed"]["fits_budget"] = bool(fits)
        del m4
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    # The suite runs several trainers back to back; their checkpoints are worth
    # nothing once the verdict is in, and each is a few GB of fp32 weights plus
    # optimizer state. The one that matters is already on the HF branch.
    for _stage in ("overfit", "resume", "pulled", "xsession"):
        shutil.rmtree(CKPT_DIR / _stage, ignore_errors=True)
    print("cleared intermediate sanity checkpoints from %s" % CKPT_DIR)

    # .get all the way down: a session that ran one stage still has to produce a
    # verdict, and it must be False for anything neither run nor carried forward
    # rather than raising or quietly reading as passed.
    _res = SANITY.get("resume") or {}
    SANITY["passed"] = bool((SANITY.get("overfit") or {}).get("passed")
                            and _res.get("in_process_passed")
                            and _res.get("cross_session_passed")
                            and (SANITY.get("speed") or {}).get("fits_budget"))
    print("")
    print("SANITY STATUS (this session's stages plus everything carried forward)")
    for _k, _v in (("overfit", (SANITY.get("overfit") or {}).get("passed")),
                   ("resume in-process", _res.get("in_process_passed")),
                   ("resume cross-session", _res.get("cross_session_passed")),
                   ("speed projection", (SANITY.get("speed") or {}).get("fits_budget"))):
        print("  %-22s %s" % (_k, "PASS" if _v else ("FAIL" if _v is False else "not run")))
    print("  %-22s %s" % ("overall", "READY for RUN_MODE='train'" if SANITY["passed"]
                          else "NOT ready - a stage above is unproven"))
    out("sanity_report.json").write_text(json.dumps(jsonable(SANITY), indent=1, sort_keys=True), encoding="utf-8")
    print("\nsaved -> %s" % out("sanity_report.json"))
    # Uploaded so the real training run - which happens in a different session -
    # can prove the sanity suite was actually run, instead of taking it on faith.
    hf_upload_file(out("sanity_report.json"), branch="main", name="sanity_report.json")
    rec("sanity", **SANITY)

## 11 — The Gate 2 comparator: zero-shot turbo, *this* notebook's decode settings

The one number Gate 2 is decided against. Notebook 03's 0.8960 cannot serve,
because it was measured with an unguarded greedy decoder that fell into
repetition loops on a handful of recordings — comparing a fine-tuned model
decoded *with* `no_repeat_ngram_size=4` against it would credit fine-tuning with
about 6 points it did not earn.

So the un-fine-tuned `whisper-large-v3-turbo` decodes both dev layers here, under
exactly the settings the fine-tuned model will be evaluated with. It costs about
15 minutes of GPU in a notebook that has to load Whisper anyway, and the result
is cached to the hub so a resumed session never pays it twice.

In [ ]:
# ============================================================
# CELL 11 - ZERO-SHOT COMPARATOR
# ============================================================
head("ZERO-SHOT COMPARATOR (un-fine-tuned %s, lang=%s, %s)"
     % (HF_MODEL, LANG_CODE, "guarded greedy"))

BASELINE = None
_cache = out("baseline_zeroshot.json")
if _cache.exists():
    BASELINE = json.loads(_cache.read_text(encoding="utf-8"))
    print("loaded the cached comparator from this session's output dir")
elif HF_ENABLED:
    # This run's repo first, then the base recipe's - a comparator is a property
    # of the decode settings, not of the training recipe, and the fingerprint
    # check below rejects it if that assumption ever stops holding.
    BASELINE = hf_fetch_json("baseline_zeroshot.json", repos=[HF_REPO, BASE_REPO])
    if BASELINE is not None:
        _cache.write_text(json.dumps(BASELINE, ensure_ascii=False, indent=1, sort_keys=True),
                          encoding="utf-8")

# A cached comparator is only usable if it was decoded exactly the way the
# fine-tuned model is about to be. Otherwise the cache silently reintroduces the
# 29 Aug bug it was meant to prevent: two sides of the Gate 2 comparison,
# different decode algorithms.
DECODE_SIG = dict(model=HF_MODEL, lang=LANG_CODE, no_repeat_ngram=NO_REPEAT_NGRAM,
                  long_mode=LONG_MODE, gen_max_length=GEN_MAX_LENGTH,
                  long_chunk_s=LONG_CHUNK_S, long_overlap_s=LONG_OVERLAP_S,
                  run_mode=RUN_MODE)
if BASELINE is not None:
    _cached_sig = {k: BASELINE.get(k) for k in DECODE_SIG}
    if _cached_sig != DECODE_SIG:
        _diff = {k: (_cached_sig.get(k), v) for k, v in DECODE_SIG.items()
                 if _cached_sig.get(k) != v}
        print("cached comparator was decoded differently - recomputing.")
        print("  changed (cached -> now): %s" % _diff)
        BASELINE = None

if BASELINE is None:
    if RUN_MODE == "sanity":
        # Enough to walk the code path, including at least two clips past the
        # 30 s window so the long-form branch is really exercised.
        base_dev_items = TINY_DEV.items
        if ODEV is not None:
            _lens = ODEV["audio_filename"].map(lambda f: (ODEV_CLIPS / f).stat().st_size)
            base_odev = ODEV.loc[_lens.sort_values(ascending=False).index[:4]].copy()
            base_odev = pd.concat([base_odev, ODEV.head(4)]).drop_duplicates("audio_filename").reset_index(drop=True)
        else:
            base_odev = None
    else:
        base_dev_items = DEV_ITEMS
        base_odev = ODEV

    t0 = time.time()
    bm = build_model(TRAIN_VARIANT, for_training=False).to(DEVICE)
    apply_gen_config(bm)
    print("base model loaded in %.0f s" % (time.time() - t0))

    print("decoding the Jember dev split...")
    base_hyp_dev = decode_jember(bm, base_dev_items, tag="zs-jember")
    m_dev, ref_f, ref_o = score_jember(base_dev_items, base_hyp_dev)

    m_odev, base_hyp_odev = {}, []
    if base_odev is not None:
        print("decoding the official dev clips...")
        base_hyp_odev = decode_official(bm, base_odev, tag="zs-official")
        m_odev = score_official(base_odev, base_hyp_odev)

    BASELINE = dict(DECODE_SIG,
                    n_dev=len(base_dev_items), n_official=len(base_odev) if base_odev is not None else 0,
                    jember=m_dev, official=m_odev,
                    dev_item_ids=[it["item_id"] for it in base_dev_items],
                    dev_hyps=base_hyp_dev,
                    official_files=(list(base_odev["audio_filename"]) if base_odev is not None else []),
                    official_hyps=base_hyp_odev,
                    decode_paths=dict(DECODE_PATHS))
    _cache.write_text(json.dumps(jsonable(BASELINE), indent=1, sort_keys=True), encoding="utf-8")
    hf_upload_file(_cache, branch="main", name="baseline_zeroshot.json")
    pd.DataFrame(dict(item_id=BASELINE["dev_item_ids"], hypothesis=base_hyp_dev)) \
        .to_csv(out("baseline_predictions_dev.csv"), index=False)
    if base_odev is not None:
        pd.DataFrame(dict(audio_filename=BASELINE["official_files"], hypothesis=base_hyp_odev)) \
            .to_csv(out("baseline_predictions_official.csv"), index=False)
    del bm
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print("")
print("COMPARATOR - Jember dev (%d items)" % BASELINE["n_dev"])
print("  folded  space : WER %.4f  (jv %.4f, id %.4f)  jv_to_id %.2f%%  len_ratio %.3f"
      % (BASELINE["jember"]["folded_wer"], BASELINE["jember"]["folded_wer_jv"],
         BASELINE["jember"]["folded_wer_id"], BASELINE["jember"]["folded_jv_to_id_sub_pct"],
         BASELINE["jember"]["folded_len_ratio"]))
print("  official space: WER %.4f   (NB 03 unguarded was %.4f - the difference is"
      % (BASELINE["jember"]["official_wer"], NB03["wer_official_unguarded"]))
print("                  what the repetition guard alone is worth)")
if BASELINE["official"]:
    print("COMPARATOR - official dev (%d clips)" % BASELINE["n_official"])
    print("  official space: WER %.4f  (jv %.4f, id %.4f)  digits in %.1f%% of hyps"
          % (BASELINE["official"]["official_wer"], BASELINE["official"]["official_wer_jv"],
             BASELINE["official"]["official_wer_id"], BASELINE["official"]["official_pct_hyp_with_digit"]))
    print("  folded space  : WER %.4f" % BASELINE["official"]["folded_wer"])
print("  decode paths  :", BASELINE["decode_paths"])

rec("baseline", **{k: v for k, v in BASELINE.items()
                   if k not in ("dev_hyps", "official_hyps", "dev_item_ids", "official_files")})

## 12 — Training

Checkpoint selection is on **dev WER**, never loss — on a seq2seq model the two
routinely disagree, and loss cannot see a repetition loop at all. Early stopping
watches the same metric.

Resumable state goes to the hub every `PUSH_EVERY_N_SAVES` saves. `RESUME_FROM_HF`
pulls it back and continues; the step counter, the optimizer moments and the
scheduler position all come from the checkpoint, not from this cell.

In [ ]:
# ============================================================
# CELL 12 - TRAINING
# ============================================================
head("TRAINING")

if RUN_MODE == "sanity":
    # Small but structurally identical: the eval hook fires, a checkpoint is
    # selected on dev WER, and the artifacts downstream are real files.
    train_ds = TINY
    eval_items = TINY_DEV.items
    eval_ds = TINY_DEV
    targs = base_targs(CKPT_DIR / "run", num_train_epochs=1, max_steps=40,
                       eval_steps=20, save_steps=20, logging_steps=10, save_total_limit=2)
else:
    train_ds = WhisperItems(TRAIN_ITEMS, TOK, MAX_LABEL_LEN)
    eval_ds  = WhisperItems(DEV_SUB_ITEMS, TOK, MAX_LABEL_LEN)
    eval_items = eval_ds.items
    print("train dataset: %d items (%d dropped for an over-long target)"
          % (len(train_ds), train_ds.n_dropped))
    print("eval  dataset: %d items" % len(eval_ds))
    targs = base_targs(CKPT_DIR / "run")

eval_refs = [it["text"] for it in eval_items]

MODEL = build_model(TRAIN_VARIANT, for_training=True).to(DEVICE)

# The projection the brief demands, on the configuration that is about to run.
PROJECTION = {}
if RUN_MODE != "sanity":
    head("PRE-FLIGHT - MEASURED SPEED AND PROJECTED WALL-CLOCK")
    _probe = speed_probe(MODEL, train_ds)
    PROJECTION = project_wallclock(len(train_ds), _probe)
    PROJECTION.update(_probe)
    if _probe["peak_vram_gb"] > 13.0:
        print("!! peak VRAM %.1f GB on a 16 GB card. Drop PER_DEVICE_BATCH to 2 and"
              % _probe["peak_vram_gb"])
        print("!! raise GRAD_ACCUM to 8 - same effective batch, far more headroom.")
    if PROJECTION["total_hours"] > MAX_TRAIN_HOURS and not ACCEPT_LONG_RUN:
        raise RuntimeError(
            "Projected %.2f h exceeds MAX_TRAIN_HOURS=%.1f. Shrink first (fewer epochs, "
            "larger GRAD_ACCUM, TRAIN_VARIANT='freeze_encoder'), or set ACCEPT_LONG_RUN=True "
            "if you really mean to run past it. A Kaggle session dies at ~12 h."
            % (PROJECTION["total_hours"], MAX_TRAIN_HOURS))
    rec("projection", **PROJECTION)

OFFICIAL_CB = OfficialEvalCallback(
    ODEV_SUB if (OFFICIAL_EVAL_DURING_TRAINING and RUN_MODE != "sanity") else
    (ODEV_SUB.head(4) if (OFFICIAL_EVAL_DURING_TRAINING and ODEV_SUB is not None) else None))
CALLBACKS = [EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE),
             PushStateCallback(), OFFICIAL_CB]
TRAINER = build_trainer(MODEL, train_ds, eval_ds, eval_refs, targs, CALLBACKS)

resume_path = None
if RESUME_FROM_HF and RUN_MODE == "train":
    pulled = hf_pull_branch(STATE_BRANCH, CKPT_DIR / "pulled")
    if pulled is not None and (pulled / "trainer_state.json").exists():
        resume_path = str(pulled)
        print("resuming from %s (global_step %d)"
              % (resume_path, json.loads((pulled / "trainer_state.json").read_text())["global_step"]))
    else:
        print("!!! RESUME_FROM_HF is on but no usable checkpoint came down. Starting fresh.")

t0 = time.time()
TRAIN_RESULT = TRAINER.train(resume_from_checkpoint=resume_path)
train_wall_h = (time.time() - t0) / 3600
print("\ntraining wall-clock: %.2f h" % train_wall_h)

# train_log.csv: step, loss, lr, dev WER, time - one row per logged event.
rows = []
for h in TRAINER.state.log_history:
    rows.append(dict(step=h.get("step"), epoch=h.get("epoch"),
                     loss=h.get("loss"), learning_rate=h.get("learning_rate"),
                     eval_wer=h.get("eval_wer"), eval_loss=h.get("eval_loss"),
                     eval_ins=h.get("eval_ins_rate"), eval_del=h.get("eval_del_rate"),
                     eval_sub=h.get("eval_sub_rate"), eval_len_ratio=h.get("eval_len_ratio"),
                     eval_runtime=h.get("eval_runtime")))
LOG = pd.DataFrame(rows)
LOG.to_csv(out("train_log.csv"), index=False)
_ev = LOG.dropna(subset=["eval_wer"])
print("\neval history (dev subset, folded space):")
print(_ev[["step", "epoch", "eval_wer", "eval_loss", "eval_len_ratio"]].to_string(index=False)
      if len(_ev) else "  (no eval rows - check eval_strategy)")

# The official curve - the whole point of this run's addition. Two questions it
# answers that the endpoint could not: did the official layer ever improve, and
# was its best step the same step Jember picked?
OFFICIAL_CURVE = pd.DataFrame(OFFICIAL_CB.rows)
BEST_OFFICIAL_STEP = None
if len(OFFICIAL_CURVE):
    OFFICIAL_CURVE.to_csv(out("official_curve.csv"), index=False)
    print("\nOFFICIAL DEV CURVE (%d clips, observation only - never selects a checkpoint)"
          % len(OFFICIAL_CB.df))
    print(OFFICIAL_CURVE[["step", "epoch", "official_wer", "official_wer_jv",
                          "official_wer_id", "official_jv_to_id"]].to_string(index=False))
    _i = OFFICIAL_CURVE["official_wer"].idxmin()
    BEST_OFFICIAL_STEP = int(OFFICIAL_CURVE.at[_i, "step"])
    print("\nbest official WER %.4f at step %d (epoch %.2f)"
          % (OFFICIAL_CURVE.at[_i, "official_wer"], BEST_OFFICIAL_STEP,
             OFFICIAL_CURVE.at[_i, "epoch"]))
    _first, _last = OFFICIAL_CURVE["official_wer"].iloc[0], OFFICIAL_CURVE["official_wer"].iloc[-1]
    if BEST_OFFICIAL_STEP != int(OFFICIAL_CURVE["step"].iloc[-1]):
        print("!! the official layer PEAKED EARLY and then regressed: %.4f at step %d"
              % (OFFICIAL_CURVE.at[_i, "official_wer"], BEST_OFFICIAL_STEP))
        print("!! but %.4f at the last step. Training past that step made the model"
              % _last)
        print("!! better on Jember and worse on the distribution that is scored.")
        print("!! Consider fewer epochs, or an earlier checkpoint, in the next run.")
    else:
        print("the official layer was still improving at the last step - it did not peak early.")
    rec("official_curve", rows=OFFICIAL_CURVE.to_dict("records"),
        best_step=BEST_OFFICIAL_STEP, first_wer=round(float(_first), 5),
        last_wer=round(float(_last), 5), n_clips=len(OFFICIAL_CB.df))
elif OFFICIAL_EVAL_DURING_TRAINING:
    print("\n!! no official curve was collected (callback disabled or no official dev)")

best_ckpt = TRAINER.state.best_model_checkpoint
best_wer = TRAINER.state.best_metric
print("\nbest checkpoint : %s" % best_ckpt)
print("best dev WER    : %s" % best_wer)

BEST_DIR = CKPT_DIR / "best"
shutil.rmtree(BEST_DIR, ignore_errors=True)
BEST_DIR.mkdir(parents=True, exist_ok=True)
TRAINER.save_model(str(BEST_DIR))          # load_best_model_at_end -> this IS the best
PROCESSOR.save_pretrained(str(BEST_DIR))
print("best weights written to %s (%.2f GB)"
      % (BEST_DIR, sum(p.stat().st_size for p in BEST_DIR.rglob("*") if p.is_file()) / 1e9))

rec("training", wall_clock_hours=round(train_wall_h, 3),
    global_step=int(TRAINER.state.global_step), epochs_done=round(float(TRAINER.state.epoch or 0), 3),
    best_checkpoint=str(best_ckpt), best_eval_wer=(round(float(best_wer), 5) if best_wer is not None else None),
    n_eval_points=int(len(_ev)), optimizer=OPTIM_USED,
    early_stopped=bool(TRAINER.state.global_step < (targs.max_steps if targs.max_steps and targs.max_steps > 0 else 10 ** 9)
                       and len(_ev) > 0 and TRAINER.state.epoch is not None
                       and float(TRAINER.state.epoch) < NUM_EPOCHS - 0.01))

## 13 — Final evaluation, both layers

The fine-tuned model, decoded once over the **full** Jember dev split and once
over the **372 official clips**, with the same settings the comparator used.

Four things get measured beyond WER, because WER alone can improve for reasons
that will not survive the test set:

- `jv_to_id` — Javanese reference words replaced by Indonesian ones. Falling WER
  with a rising `jv_to_id` means the Indonesian half improved while the model got
  *more* eager to translate away the Javanese.
- **Cosmetic error share** — errors that are only diacritics, case, or
  punctuation. Against zero-shot this should collapse; if it does not, something
  is wrong with the training text.
- **Length ratio** — hypothesis tokens over reference tokens. Early warning for
  both hallucination and truncation.
- **Digits** — the official reference never writes a numeral. Zero-shot emitted
  `40` for `empat puluh`; a fine-tuned model should have stopped.

In [ ]:
# ============================================================
# CELL 13 - FINAL EVALUATION
# ============================================================
head("FINAL EVALUATION")

EVAL_MODEL = TRAINER.model
if TRAIN_VARIANT == "lora":
    try:
        EVAL_MODEL = EVAL_MODEL.merge_and_unload()
        print("LoRA adapters merged for inference")
    except Exception as e:
        print("could not merge LoRA adapters (%r) - evaluating the wrapped model" % (e,))
EVAL_MODEL = EVAL_MODEL.to(DEVICE)
if DEVICE == "cuda":
    EVAL_MODEL = EVAL_MODEL.half()          # inference only; training is done
EVAL_MODEL.eval()
apply_gen_config(EVAL_MODEL)
EVAL_MODEL.config.use_cache = True

fin_dev_items = TINY_DEV.items if RUN_MODE == "sanity" else DEV_ITEMS
fin_odev = None
if ODEV is not None:
    fin_odev = ODEV if RUN_MODE != "sanity" else \
        ODEV[ODEV["audio_filename"].isin(BASELINE["official_files"])].reset_index(drop=True)

print("Jember dev: %d items" % len(fin_dev_items))
FT_HYP_DEV = decode_jember(EVAL_MODEL, fin_dev_items, tag="ft-jember")
M_DEV, REF_F, REF_O = score_jember(fin_dev_items, FT_HYP_DEV)
pd.DataFrame(dict(seg_id=[it["item_id"] for it in fin_dev_items], hypothesis=FT_HYP_DEV)) \
    .to_csv(out("predictions_dev.csv"), index=False)

M_ODEV, FT_HYP_ODEV = {}, []
if fin_odev is not None and len(fin_odev):
    print("official dev: %d clips" % len(fin_odev))
    # Same order as the comparator, so the paired bootstrap in cell 14 lines up.
    fin_odev = fin_odev.set_index("audio_filename").loc[
        [f for f in BASELINE["official_files"] if f in set(fin_odev["audio_filename"])]
    ].reset_index() if BASELINE["official_files"] else fin_odev
    FT_HYP_ODEV = decode_official(EVAL_MODEL, fin_odev, tag="ft-official")
    M_ODEV = score_official(fin_odev, FT_HYP_ODEV)
    pd.DataFrame(dict(audio_filename=list(fin_odev["audio_filename"]), hypothesis=FT_HYP_ODEV)) \
        .to_csv(out("predictions_official_dev.csv"), index=False)

def show(title, m, b, keys):
    print("\n%s" % title)
    print("  %-24s %10s %10s %10s" % ("metric", "zero-shot", "fine-tuned", "delta"))
    for k in keys:
        if k not in m or k not in b:
            continue
        print("  %-24s %10.4f %10.4f %+10.4f" % (k, b[k], m[k], m[k] - b[k]))

KEYS = ["folded_wer", "folded_wer_jv", "folded_wer_id", "folded_sub_rate", "folded_del_rate",
        "folded_ins_rate", "folded_jv_to_id_sub_pct", "folded_cosmetic_sub_pct",
        "folded_len_ratio", "official_wer", "official_wer_jv", "official_wer_id",
        "official_pct_hyp_with_digit"]
show("JEMBER DEV - zero-shot vs fine-tuned", M_DEV, BASELINE["jember"], KEYS)
if M_ODEV:
    show("OFFICIAL DEV (PRIMARY LAYER) - zero-shot vs fine-tuned", M_ODEV, BASELINE["official"], KEYS)

# Error tables: what the model actually gets wrong, tagged cosmetic/not.
ET = scorer.error_table([fold(r) for r in REF_F], [fold(h) for h in FT_HYP_DEV], LEX, top_k=80)
(ET if hasattr(ET, "to_csv") else pd.DataFrame(ET)).to_csv(out("error_table_dev.csv"), index=False)
print("\ntop Jember dev confusions (folded space):")
print(ET.head(15).to_string(index=False) if hasattr(ET, "head") else ET[:15])
if M_ODEV:
    ETO = scorer.error_table(list(fin_odev["ref_official"]), FT_HYP_ODEV, LEX, top_k=80)
    (ETO if hasattr(ETO, "to_csv") else pd.DataFrame(ETO)).to_csv(out("error_table_official_dev.csv"), index=False)
    print("\ntop official dev confusions:")
    print(ETO.head(15).to_string(index=False) if hasattr(ETO, "head") else ETO[:15])

# Per-recording spread on the Jember dev - one bad recording can carry a result.
_pr = pd.DataFrame(dict(rec=[it["rec"] for it in fin_dev_items],
                        ref=[fold(r) for r in REF_F], hyp=[fold(h) for h in FT_HYP_DEV]))
PER_REC = pd.DataFrame([dict(recording=r, n=len(g),
                             wer=round(scorer.wer(list(g["ref"]), list(g["hyp"]))["wer"], 4))
                        for r, g in _pr.groupby("rec")]).sort_values("wer")
print("\nper-recording WER (folded space), std %.4f:" % PER_REC["wer"].std())
print(PER_REC.to_string(index=False))

EVAL_FINAL = dict(run_id=RUN_ID, run_mode=RUN_MODE, model=HF_MODEL, variant=TRAIN_VARIANT,
                  segment_mode=SEGMENT_MODE, lang=LANG_CODE,
                  jember=M_DEV, official=M_ODEV,
                  baseline_jember=BASELINE["jember"], baseline_official=BASELINE["official"],
                  n_dev=len(fin_dev_items), n_official=(len(fin_odev) if fin_odev is not None else 0),
                  per_recording_wer_std=round(float(PER_REC["wer"].std()), 4),
                  decode_paths=dict(DECODE_PATHS))
out("eval_final.json").write_text(json.dumps(jsonable(EVAL_FINAL), indent=1, sort_keys=True), encoding="utf-8")
print("\nsaved -> %s" % out("eval_final.json"))
rec("eval_final", jember=M_DEV, official=M_ODEV,
    per_recording_wer_std=EVAL_FINAL["per_recording_wer_std"], decode_paths=dict(DECODE_PATHS))

## 14 — Gate 2

Notebook 02 measured this dev set's resolution: a WER gap under roughly 1.5
points is not proven by a single run. So "better" here is not a smaller number —
it is a **paired bootstrap** over the same utterances whose 95 % interval stays
on one side of zero (`scorer.compare_systems`, CPU, free).

The gate, as the brief writes it:

1. fine-tuned dev WER **better** than the notebook-03-style zero-shot best
2. Javanese WER improves, or the reason it did not is understood and written down
3. `jv_to_id` no worse than the zero-shot baseline
4. checkpoint-resume proven to work
5. best checkpoint on a private HF repo with the revision recorded

Failing (1) means **do not go to notebook 06.** Fine-tuning that fails on 8.5
hours of in-domain data is a label problem, a normalisation problem, or too
large a learning rate — almost never a shortage of data.

In [ ]:
# ============================================================
# CELL 14 - GATE 2
# ============================================================
head("GATE 2")

def paired(refs, hyps_zero, hyps_ft, label):
    c = scorer.compare_systems(refs, hyps_zero, hyps_ft, n_boot=2000, seed=SEED)
    better = c["delta"] < 0
    proven = c["delta_hi"] < 0            # the whole 95 % interval is an improvement
    print("  %-22s zero-shot %.4f -> fine-tuned %.4f   delta %+.4f  95%% CI [%+.4f, %+.4f]  p(better) %.3f"
          % (label, c["wer_a"], c["wer_b"], c["delta"], c["delta_lo"], c["delta_hi"], c["p_b_better"]))
    print("  %-22s %s" % ("", "PROVEN improvement" if proven else
                          ("improved, but the interval still contains zero - not proven from one run"
                           if better else "NOT an improvement")))
    return dict(label=label, wer_zero=round(c["wer_a"], 4), wer_ft=round(c["wer_b"], 4),
                delta=round(c["delta"], 4), delta_lo=round(c["delta_lo"], 4),
                delta_hi=round(c["delta_hi"], 4), p_better=round(c["p_b_better"], 3),
                better=bool(better), proven=bool(proven))

PAIRED = {}

# --- Jember dev, folded space (primary for this layer) ---
_bmap = dict(zip(BASELINE["dev_item_ids"], BASELINE["dev_hyps"]))
_ids = [it["item_id"] for it in fin_dev_items]
_common = [i for i in _ids if i in _bmap]
if len(_common) != len(_ids):
    print("  note: %d of %d dev items are missing from the cached comparator; comparing on the overlap"
          % (len(_ids) - len(_common), len(_ids)))
_pos = {i: k for k, i in enumerate(_ids)}
_refs_f = [fold(REF_F[_pos[i]]) for i in _common]
_zero_f = [fold(_bmap[i]) for i in _common]
_ft_f   = [fold(FT_HYP_DEV[_pos[i]]) for i in _common]
print("Jember dev, folded space (n=%d):" % len(_common))
PAIRED["jember_folded"] = paired(_refs_f, _zero_f, _ft_f, "jember/folded")
_refs_o = [REF_O[_pos[i]] for i in _common]
PAIRED["jember_official"] = paired(_refs_o, [_bmap[i] for i in _common],
                                   [FT_HYP_DEV[_pos[i]] for i in _common], "jember/official")

# --- Official dev, the layer that predicts the test set ---
if M_ODEV and BASELINE["official"]:
    _omap = dict(zip(BASELINE["official_files"], BASELINE["official_hyps"]))
    _of = [f for f in fin_odev["audio_filename"] if f in _omap]
    _opos = {f: k for k, f in enumerate(fin_odev["audio_filename"])}
    _oref = list(fin_odev["ref_official"])
    print("\nOfficial dev, official space (n=%d) - THE PRIMARY LAYER:" % len(_of))
    PAIRED["official"] = paired([_oref[_opos[f]] for f in _of],
                                [_omap[f] for f in _of],
                                [FT_HYP_ODEV[_opos[f]] for f in _of], "official/official")

# --- the five criteria ---
head("GATE 2 CRITERIA")
SANITY_REPORT = SANITY if SANITY else None
if SANITY_REPORT is None:
    # A training run happens in a different session from the sanity run, so the
    # report is fetched rather than assumed.
    if out("sanity_report.json").exists():
        SANITY_REPORT = json.loads(out("sanity_report.json").read_text(encoding="utf-8"))
    else:
        # Looks in the sanity run's own repo too - a training run is a different
        # RUN_ID and therefore a different repo from the run that proved resume.
        SANITY_REPORT = hf_fetch_json("sanity_report.json")
        if SANITY_REPORT is None:
            print("no sanity_report.json found locally or in either hub repo")

resume_proven = bool(SANITY_REPORT and SANITY_REPORT.get("resume", {}).get("in_process_passed")
                     and SANITY_REPORT.get("resume", {}).get("cross_session_passed"))

primary = PAIRED.get("official") or PAIRED["jember_folded"]
jv_zero = (BASELINE["official"] if M_ODEV else BASELINE["jember"])
jv_ft   = (M_ODEV if M_ODEV else M_DEV)
jv_key  = "official_jv_to_id_sub_pct" if M_ODEV else "folded_jv_to_id_sub_pct"
wjv_key = "official_wer_jv" if M_ODEV else "folded_wer_jv"

CRIT = []
def crit(name, ok, detail=""):
    CRIT.append(dict(criterion=name, passed=bool(ok), detail=str(detail)[:200]))
    print("  [%s] %-52s %s" % ("PASS" if ok else "FAIL", name, detail))
    return bool(ok)

crit("1. fine-tuned dev WER better than zero-shot (paired, proven)",
     primary["proven"],
     "%s: %.4f -> %.4f, CI [%+.4f, %+.4f]" % (primary["label"], primary["wer_zero"],
                                              primary["wer_ft"], primary["delta_lo"], primary["delta_hi"]))
crit("2. Javanese WER improves", jv_ft.get(wjv_key, 9) <= jv_zero.get(wjv_key, 0),
     "wer_jv %.4f -> %.4f" % (jv_zero.get(wjv_key, float("nan")), jv_ft.get(wjv_key, float("nan"))))
crit("3. jv_to_id no worse than zero-shot",
     jv_ft.get(jv_key, 100) <= jv_zero.get(jv_key, 0) + 0.01,
     "%.2f%% -> %.2f%%  (NB 03 historical %.2f%%)"
     % (jv_zero.get(jv_key, float("nan")), jv_ft.get(jv_key, float("nan")), NB03["jv_to_id_sub_pct"]))
crit("4. checkpoint-resume proven (in-process AND cross-session)", resume_proven,
     (SANITY_REPORT or {}).get("resume", {}))
crit("5. best checkpoint on private HF with a recorded revision", HF_ENABLED,
     HF_REPO or "hub disabled")   # the revision itself is written in cell 15

n_pass = sum(1 for c in CRIT if c["passed"])
GATE2 = dict(run_id=RUN_ID, run_mode=RUN_MODE, criteria=CRIT, n_passed=n_pass, n_total=len(CRIT),
             paired=PAIRED, primary_layer=primary["label"], resume_proven=resume_proven)
print("")
if RUN_MODE == "sanity":
    print(">>> SANITY RUN - this is the gate ARITHMETIC walking its code path on a")
    print(">>> handful of clips. It is not a verdict. Set RUN_MODE = 'train'.")
    GATE2["verdict"] = "sanity"
elif n_pass == len(CRIT):
    print(">>> GATE 2 PASSED. Notebook 06 (data mixing) may start.")
    GATE2["verdict"] = "passed"
elif not CRIT[0]["passed"]:
    print(">>> GATE 2 BLOCKED on criterion 1. DO NOT go to notebook 06.")
    print(">>> Fine-tuning that fails on 8.5 h of in-domain data is a label problem,")
    print(">>> a normalisation problem, or too large a learning rate. Audit in that")
    print(">>> order: re-read cell 10's overfit result, then the training text, then lr.")
    GATE2["verdict"] = "blocked"
else:
    print(">>> GATE 2 CONDITIONAL: the WER criterion passed but %d other criteria did not."
          % (len(CRIT) - n_pass))
    print(">>> Read each FAIL above. A failure on 2 or 3 means the gain came from the")
    print(">>> Indonesian half and is unlikely to survive the test set.")
    GATE2["verdict"] = "conditional"

out("gate2.json").write_text(json.dumps(jsonable(GATE2), indent=1, sort_keys=True), encoding="utf-8")
rec("gate2", **GATE2)

## 15 — Push the best checkpoint, write the manifest

Weights never travel through the user's laptop. `main` gets the best weights and
the processor; `checkpoint_manifest.json` records the repo, the revision, the
size on disk and a hash per file, so a later notebook can prove it loaded the
same artifact this notebook measured.

In [ ]:
# ============================================================
# CELL 15 - PUSH + CHECKPOINT MANIFEST
# ============================================================
head("CHECKPOINT")

files = sorted(p for p in BEST_DIR.rglob("*") if p.is_file())
size_bytes = sum(p.stat().st_size for p in files)
hashes = {}
for p in files:
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 22), b""):
            h.update(chunk)
    hashes[str(p.relative_to(BEST_DIR))] = h.hexdigest()[:16]
    print("  %-32s %8.1f MB  %s" % (p.name, p.stat().st_size / 1e6, hashes[str(p.relative_to(BEST_DIR))]))

revision = None
if HF_ENABLED:
    revision = hf_push_folder(BEST_DIR, "main", "%s best checkpoint (dev WER %s)"
                              % (RUN_ID, TRAINER.state.best_metric))

CKPT_MANIFEST = dict(
    run_id=RUN_ID, run_mode=RUN_MODE, base_model=HF_MODEL, variant=TRAIN_VARIANT,
    segment_mode=SEGMENT_MODE, lang=LANG_CODE,
    hf_repo=HF_REPO, hf_revision=revision, hf_branch="main",
    state_branch=STATE_BRANCH, private=True,
    size_bytes=int(size_bytes), size_gb=round(size_bytes / 1e9, 3),
    n_files=len(files), sha256_16=hashes,
    best_local_checkpoint=str(TRAINER.state.best_model_checkpoint),
    best_eval_wer=(round(float(TRAINER.state.best_metric), 5) if TRAINER.state.best_metric is not None else None),
    global_step=int(TRAINER.state.global_step),
    optimizer=OPTIM_USED, effective_batch=PER_DEVICE_BATCH * GRAD_ACCUM,
    learning_rate=LEARNING_RATE, num_epochs=NUM_EPOCHS,
    decode=dict(no_repeat_ngram_size=NO_REPEAT_NGRAM, num_beams=1, max_length=GEN_MAX_LENGTH))
out("checkpoint_manifest.json").write_text(
    json.dumps(jsonable(CKPT_MANIFEST), indent=1, sort_keys=True), encoding="utf-8")
print("\nrepo     : %s" % HF_REPO)
print("revision : %s" % revision)
print("size     : %.2f GB across %d files" % (size_bytes / 1e9, len(files)))
print("saved -> %s" % out("checkpoint_manifest.json"))
if not HF_ENABLED:
    print("\n!!! NOT PUSHED - the hub is disabled. These weights die with the session.")
hf_upload_file(out("checkpoint_manifest.json"), branch="main")
hf_upload_file(out("eval_final.json"), branch="main")
rec("checkpoint", **{k: v for k, v in CKPT_MANIFEST.items() if k != "sha256_16"})

## 16 — Self-validation

Each check is a way this notebook could have produced numbers that look fine and
mean nothing. They map onto the brief's traps and completion criteria.

In [ ]:
# ============================================================
# CELL 16 - SELF-VALIDATION
# ============================================================
head("SELF-VALIDATION")
CHECKS = []
def check(name, cond, detail=""):
    CHECKS.append(dict(check=name, passed=bool(cond), detail=str(detail)[:180]))
    print("  [%s] %-60s %s" % ("PASS" if cond else "FAIL", name, detail))
    return bool(cond)

# Reconstructed from the audio window, not from the item id. In concat25 EVERY
# item carries a synthesised id ("1_c210") - including a group that happens to
# hold one segment - so looking the id up in the manifest raised KeyError and,
# worse, the old version of this check only ever looked at n_seg == 1 items and
# would have silently skipped every real concatenation. This form verifies the
# actual claim in both modes: the target text is the text_folded of exactly the
# segments the item's sample window covers.
_ok_target, _n_checked, _bad = True, 0, None
for it in TRAIN_ITEMS[:50]:
    g = MANIFEST[(MANIFEST["audio_filename"] == it["rec"]) &
                 (MANIFEST["start_sample"] >= it["start"]) &
                 (MANIFEST["end_sample"] <= it["end"])].sort_values("start_sample")
    if " ".join(g["text_folded"]).strip() != it["text"] or len(g) != it["n_seg"]:
        _ok_target, _bad = False, it["item_id"]
        break
    _n_checked += 1
check("training target is text_folded of exactly the segments it covers",
      _ok_target, "%d items verified against the manifest window%s"
      % (_n_checked, "" if _ok_target else "; first mismatch: %s" % _bad))
check("label prefix carried the <|%s|> token" % LANG_CODE,
      any("<|%s|>" % LANG_CODE == t for t in FINDINGS["model_setup"]["label_prefix"]),
      FINDINGS["model_setup"]["label_prefix"])
check("startoftranscript was trimmed from the labels",
      bool(SANITY.get("collator", {}).get("prefix_trimmed", RUN_MODE != "sanity")),
      "asserted in the collator on every batch")
check("the comparator was measured here, not taken from NB 03",
      BASELINE["no_repeat_ngram"] == NO_REPEAT_NGRAM and BASELINE["model"] == HF_MODEL,
      "zero-shot %.4f (folded) vs NB 03's unguarded %.4f"
      % (BASELINE["jember"]["folded_wer"], NB03["wer_folded_unguarded"]))
check("the official 372-clip layer was actually evaluated",
      bool(M_ODEV), "%d clips" % (EVAL_FINAL["n_official"]))
check("clips over 30 s were chunked, not truncated",
      DECODE_PATHS["long_form"] + DECODE_PATHS["chunk"] > 0 or ODEV is None,
      dict(DECODE_PATHS))
# The check that was missing on 29 Aug: the zero-shot pass decoded 4 long clips
# with long_form while the fine-tuned pass chunked the SAME clips, and nothing
# noticed. Comparing two systems that used different decode algorithms is not
# comparing two systems.
_bp = {k: v for k, v in (BASELINE.get("decode_paths") or {}).items() if k != "short"}
_fp = {k: v for k, v in dict(DECODE_PATHS).items() if k != "short"}
check("both Gate 2 passes used the same long-clip decode path",
      _bp == _fp or not M_ODEV,
      "zero-shot %s vs fine-tuned %s" % (_bp, _fp))
check("no long-clip decode fell back mid-run",
      DECODE_PATHS["long_fallback"] == 0,
      "LONG_MODE=%s, %d fallbacks" % (LONG_MODE, DECODE_PATHS["long_fallback"]))
check("checkpoint selection used dev WER, not loss",
      str(getattr(targs, "metric_for_best_model", "")) in ("wer", "eval_wer"),
      getattr(targs, "metric_for_best_model", None))
check("more than one eval point exists (a curve, not a single dot)",
      len(_ev) >= 2, "%d eval points" % len(_ev))
check("the official layer has a curve, not just an endpoint",
      len(OFFICIAL_CURVE) >= 2 or not OFFICIAL_EVAL_DURING_TRAINING or ODEV is None,
      "%d official measurements during training" % len(OFFICIAL_CURVE))
if BEST_OFFICIAL_STEP is not None and TRAINER.state.best_model_checkpoint:
    _sel = int(str(TRAINER.state.best_model_checkpoint).rsplit("-", 1)[-1])
    check("Jember-selected checkpoint is also the official layer's best",
          _sel == BEST_OFFICIAL_STEP,
          "Jember picked step %d, official best was step %d%s"
          % (_sel, BEST_OFFICIAL_STEP,
             "" if _sel == BEST_OFFICIAL_STEP else
             " - the selection metric and the scored distribution disagree"))
check("dev was native segments in every mode",
      all(it["n_seg"] == 1 for it in fin_dev_items), "segment_mode=%s affects training only" % SEGMENT_MODE)
check("train and dev recordings stayed disjoint",
      set(it["rec"] for it in TRAIN_ITEMS).isdisjoint(set(it["rec"] for it in fin_dev_items)))
check("best checkpoint pushed to a private HF repo with a revision",
      bool(CKPT_MANIFEST["hf_revision"]), CKPT_MANIFEST["hf_revision"])
check("checkpoint-resume proven, in-process and cross-session",
      bool(GATE2["resume_proven"]), (SANITY_REPORT or {}).get("resume", {}))
# This check used to look only inside OUT_DIR, so it passed happily while
# /kaggle/working/ckpt held 16 GB of checkpoints. The download path is the whole
# of /kaggle/working, so that is what gets measured.
_dl = [p for p in WORKING.rglob("*") if p.is_file()]
_dl_bytes = sum(p.stat().st_size for p in _dl)
_weights = [p for p in _dl if p.suffix in (".safetensors", ".bin", ".pt", ".pth")]
check("no model weights anywhere in the download path",
      not _weights,
      "%d weight files%s" % (len(_weights),
          (": " + ", ".join(str(p.relative_to(WORKING)) for p in _weights[:3])) if _weights else ""))
check("total download stays small (weights live on HF)",
      _dl_bytes < 500e6,
      "%.1f MB across %d files in %s" % (_dl_bytes / 1e6, len(_dl), WORKING))
check("not a sanity run", RUN_MODE != "sanity",
      "RUN_MODE=%s%s" % (RUN_MODE, " - expected to FAIL on a sanity run" if RUN_MODE == "sanity" else ""))

n_pass = sum(1 for c in CHECKS if c["passed"])
print("\n%d/%d checks passed" % (n_pass, len(CHECKS)))
sanity_only = (RUN_MODE == "sanity" and
               all(c["passed"] for c in CHECKS if c["check"] not in
                   ("not a sanity run", "more than one eval point exists (a curve, not a single dot)",
                    "the official 372-clip layer was actually evaluated")))
if sanity_only:
    print(">>> SANITY RUN OK. The pipeline runs end to end: labels overfit, resume")
    print(">>> restores state, the projection fits, and every artifact was written.")
    print(">>> Next: a fresh session with RESUME_FROM_HF = True to close the")
    print(">>> cross-session half of the resume proof, then RUN_MODE = 'train'.")
elif n_pass == len(CHECKS):
    print(">>> ALL CHECKS PASSED.")
else:
    print(">>> Read every FAIL above before treating any number here as final.")

rec("validation", checks=CHECKS, n_passed=n_pass, n_total=len(CHECKS),
    sanity_run=(RUN_MODE == "sanity"))

## 17 — Digest and packaging

Run last. `findings/<run id>/` is all small CSV/JSON — the weights live on
HuggingFace and never enter the download path — so there is no phase 2/3 split
here and nothing to trim before downloading.

In [ ]:
# ============================================================
# CELL 17 - DIGEST + PACKAGING
# Copy the ENTIRE block between the BEGIN/END markers back into the chat.
# ============================================================
head("ARTIFACT FINGERPRINTS")
FP = {}
for p in sorted(OUT_DIR.rglob("*.csv")) + sorted(OUT_DIR.rglob("*.json")):
    if p.is_file():
        rel = str(p.relative_to(OUT_DIR))
        FP[rel] = hashlib.sha256(p.read_bytes()).hexdigest()[:16]
        print("  %-40s %s   %8.1f KB" % (rel, FP[rel], p.stat().st_size / 1024))
rec("fingerprints", **FP)

EXPECTED = ["env", "recipe", "scorer", "data", "items", "audio", "model_setup", "optim",
            "baseline", "training", "eval_final", "gate2", "checkpoint", "validation"]
missing_sections = [k for k in EXPECTED if k not in FINDINGS]
if missing_sections:
    print("")
    print("!" * 72)
    print("INCOMPLETE DIGEST - missing sections: %s" % missing_sections)
    print("Almost always a partial re-run: cell 1 resets FINDINGS, so any cell not")
    print("executed after it contributes nothing. FIX: Run All.")
    print("!" * 72)

FINDINGS["_meta"] = dict(
    notebook="mozilla-javanese whisper fine-tune (core)",
    notebook_id=NOTEBOOK_ID, run_id=RUN_ID, run_mode=RUN_MODE,
    generated_for="handoff to Claude - do not edit by hand",
    complete=(not missing_sections), missing_sections=missing_sections,
    artifacts=sorted(str(p.relative_to(OUT_DIR)) for p in OUT_DIR.rglob("*") if p.is_file()))

blob = json.dumps(jsonable(FINDINGS), ensure_ascii=False, separators=(",", ":"), sort_keys=True)
print("")
print("===== BEGIN CLAUDE-HANDOFF-JSON =====")
print(blob)
print("===== END CLAUDE-HANDOFF-JSON =====")
print("")
print("size: %.1f KB" % (len(blob) / 1024))

try:
    out("dataset_findings.json").write_text(
        json.dumps(jsonable(FINDINGS), ensure_ascii=False, indent=1, sort_keys=True), encoding="utf-8")
    print("saved ->", out("dataset_findings.json"))
    sys.stdout.flush()
    print("saved ->", LOG_PATH)
except Exception as e:
    print("could not save:", e)

# One-file download, the NB 02 "-lite.zip" convention. Kaggle's output browser
# downloads a single file or the entire tree, nothing in between, so a folder
# only becomes downloadable by becoming a file. It is written to the working
# ROOT, not inside OUT_DIR, or it would be trying to contain itself.
try:
    _zip = shutil.make_archive(str(WORKING / (RUN_ID + "-lite")), "zip",
                               root_dir=str(OUT_DIR.parent), base_dir=RUN_ID)
    print("\none-file download: %s  (%.1f KB)" % (_zip, Path(_zip).stat().st_size / 1024))
except Exception as e:
    print("could not build the lite zip: %r" % (e,))

head("NEXT STEPS, BY HAND")
if RUN_MODE == "sanity" and not RESUME_FROM_HF:
    print("1. This was the sanity run. Read the three verdicts above: overfit,")
    print("   in-process resume, projection.")
    print("")
    print("2. Close this session completely. Start a FRESH one, set")
    print("      RESUME_FROM_HF = True")
    print("   and Run All again. That closes the cross-session half of the resume")
    print("   proof - the half that matters, because it is the one a dead 12 h")
    print("   session actually exercises.")
    print("")
    print("3. Then set RUN_MODE = 'train', RESUME_FROM_HF = False, and Run All.")
elif RUN_MODE == "sanity":
    print("1. Cross-session resume is now proven. Set RUN_MODE = 'train',")
    print("   RESUME_FROM_HF = False, and Run All in a fresh session.")
    print("")
    print("2. If that session dies mid-training, come back with RUN_MODE = 'train'")
    print("   and RESUME_FROM_HF = True - the training-state branch has the last")
    print("   pushed checkpoint.")
else:
    print("1. Output tab -> Download all. Small CSV/JSON only; the weights are on")
    print("   HuggingFace (%s)." % HF_REPO)
    print("")
    print("2. Unzip onto the repo root - it already contains findings/%s/." % RUN_ID)
    print("")
    print("3. Paste the CLAUDE-HANDOFF-JSON block above into the chat.")
    print("")
    print("4. Gate 2 verdict: %s. %s" % (GATE2["verdict"].upper(),
          "Write Hasil Notebook 04 - Whisper Core.md, then notebook 06 may start."
          if GATE2["verdict"] == "passed" else
          "Do NOT start notebook 06 until criterion 1 passes."))
    print("")
    print("5. Ablations, one variable per run, each into its own findings folder:")
    print("   TRAIN_VARIANT = 'freeze_encoder' | 'freeze_encoder_half' | 'lora'")
    print("   SEGMENT_MODE  = 'concat25'   (the 25 s vs 5 s length mismatch)")